# GP02C Trusted Pytest Adapter

Exercise the trusted pytest adapter and its provenance checks.

This source copy comes from **GP02C_03_Trusted_Real_Pytest_Live_Adapter_CPU_v1_1.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

**Clear-text source release:** No embedded ZIP, Base64 or hex payloads are required to read the notebook source.


### Preserved execution evidence — source archive

- **Executed source:** `Paper/GP02/GP02C/gp02c-03.ipynb` in the author-provided `Paper (2)(2).zip` (10 November 2026).
- **Saved results:** 7 human-readable Jupyter output records restored from matching executed cells.
- **Source fidelity:** 7 identical code cells; 0 proven value-equivalent readable source strings; 0 historical configuration cell(s) restored to align with archived execution.
- **Excluded display artifacts:** 0 active, encoded, or binary browser-download outputs were omitted; the source archive retains them. These are *not* replacements for the sealed experiment ZIPs.
- **Reproduction distinction:** Saved historical output does **not** mean this notebook has been rerun in the current environment. The original experiment may require Kaggle mounts, GPUs, offline wheels and authorised archives. The additional execution guide above details the prerequisites.


## How to execute or reproduce this implementation stage

**Execution mode:** CPU.  
**Prerequisites:** Python and available pytest with the public synthetic fixture; run in a clean notebook workspace.

**Steps:** Run all cells in order to write the required modules and execute trusted adapter acceptance tests. Inspect receipt provenance and test qualification.

**Reproduction boundary:** Runnable mechanism study; may require compatible pinned dependencies.

> **Historical evidence:** This release retains implementation source but removes historical cell outputs. Results from stochastic GPU runs are not guaranteed to repeat byte-for-byte. Use the sealed evidence records and the E17 reviewer companion to reconstruct accepted historical outcomes.


# GP02C-03 v1.1 — Real Visible-`pytest` Adapter and Deadline Smoke Test

**Research:** *When Passing Tests Become Stale: Evidence Freshness in Gemma 4 Software Repair Agents*  
**Environment:** Kaggle CPU / No Accelerator, or any local Python environment with `pytest`.  
**Scope:** trusted visible-test execution on small **synthetic** Python source trees. **No Gemma calls, no GPUs, no hidden evaluator, no competition task setup.**  
**Implementation:** GP02C-02 v1.2 ledger stays unchanged; this notebook adds a real source-snapshot + JUnit + hard subprocess-deadline adapter.

**Acceptance boundary:** This is a CPU live-`pytest` integration smoke test, **not yet Google ADK / Gemma worker wiring**. The 840/900-second defaults are implemented in the absolute deadline policy and demonstrated using scaled short test deadlines. Full v2.7 ADK integration, model-request cancellation and repository-specific dependency fingerprints are GP02C-04 work. The isolated source copy and permission hardening are not a security sandbox for malicious test code.

**Instructions:** Run the notebook top to bottom, on CPU. No data attachment required. Outputs and evidence ZIP will be written to the working directory. Each code-cell output is followed by a short observation guide.

**v1.1 corrective update:** (1) C batch feedback reconciles check states after all rechecks, with final revision/workspace SHA; (2) real and simulated receipts are qualified against budget/deadline before deriving observed regression events; original failed raw observations remain in the audit. Three extra regression tests added.


## 01 — Versioned controller v1.3 (derived from GP02C-02 v1.2)

Write the v1.2 ledger with **two GP02C-03 v1.1 corrections**: final-state batch feedback and qualification-before-regression measurement. The previous v1.2 artifact is retained separately and unchanged.


In [2]:
%%writefile gp02c_ledger.py
"""GP02C-02: CPU-only evidence-freshness ledger and B/C controller prototype.

This module does NOT run shell commands, use a model, access hidden evaluators,
or integrate with GP02B. A trusted adapter supplies visible-test results.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Mapping, Sequence
import hashlib
import json
import os
import re
import time

SCHEMA = "gp02c.ledger.v1.2"
ARMS = {"A", "B", "C"}
PROVENANCES = {"agent", "controller_recheck", "final_gate", "common_initial"}
EXCLUDED_DIRS = frozenset({
    ".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache",
    ".ruff_cache", ".tox", ".nox", ".cache", "node_modules", ".ipynb_checkpoints",
    ".gp02c_results", "__ledger_output__", "htmlcov", ".coverage-data",
})
EXCLUDED_NAMES = frozenset({".coverage", ".DS_Store", "coverage.xml"})
EXCLUDED_SUFFIXES = frozenset({".pyc", ".pyo"})
ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,120}$")


def canonical_hash(obj: object) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=True,
                                      separators=(",", ":"), default=str).encode()).hexdigest()


def utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat()


def environment_fingerprint(config: Mapping[str, object]) -> str:
    """Hash explicit environment identity supplied by the trusted harness.

    In live integration, include installed wheel/source hashes and relevant runtime
    settings, not just package names, and refresh after environment modifications.
    """
    if not config:
        raise ValueError("Nonempty trusted environment manifest required")
    return canonical_hash(dict(config))


@dataclass(frozen=True)
class Snapshot:
    sha256: str
    files: int
    entries: tuple[tuple[str, str, str], ...]


class WorkspaceFingerprinter:
    """Conservative all-files policy, except a fixed generated-artifact exclusion list.

    Includes untracked source and config; file additions, deletions and symlink
    targets affect the fingerprint. NEVER walk an evaluator/protected workspace.
    """
    def __init__(self, root: str | Path):
        self.root = Path(root).resolve(strict=True)
        if not self.root.is_dir():
            raise ValueError("Workspace root must be a directory")

    def snapshot(self) -> Snapshot:
        entries: list[tuple[str, str, str]] = []
        for base, dirs, files in os.walk(self.root, followlinks=False):
            basedir = Path(base)
            # Symlink directories are recorded as links, never dereferenced.
            for name in list(dirs):
                full = basedir / name
                if name in EXCLUDED_DIRS:
                    dirs.remove(name)
                elif full.is_symlink():
                    dirs.remove(name)
                    rel = full.relative_to(self.root).as_posix()
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
            for name in files:
                if name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                    continue
                full = basedir / name
                rel = full.relative_to(self.root).as_posix()
                if full.is_symlink():
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
                elif full.is_file():
                    h = hashlib.sha256()
                    with full.open("rb") as f:
                        for block in iter(lambda: f.read(256 * 1024), b""):
                            h.update(block)
                    entries.append((rel, "F", h.hexdigest()))
        entries.sort()
        return Snapshot(canonical_hash(entries), len(entries), tuple(entries))


@dataclass(frozen=True)
class VisibleCheck:
    check_id: str
    command: tuple[str, ...]
    definition_sha256: str
    reserve_seconds: float = 0.0
    origin: str = "visible_repo"
    working_dir: str = "."
    test_selection: str = ""
    expected_collected: int | None = None
    pinned_environment_sha256: str | None = None

    def __post_init__(self):
        if not ID_RE.fullmatch(self.check_id):
            raise ValueError("Unsafe visible check identifier")
        if self.origin != "visible_repo":
            raise ValueError("Only trusted visible-repository checks are allowed")
        if not self.command or not all(isinstance(x, str) and x for x in self.command):
            raise ValueError("Nonempty command tokens required")
        if not re.fullmatch(r"[0-9a-f]{64}", self.definition_sha256):
            raise ValueError("Check definition must have SHA-256 identity")
        if self.reserve_seconds < 0:
            raise ValueError("Negative estimate")
        if (Path(self.working_dir).is_absolute() or
            any(x == ".." for x in Path(self.working_dir).parts)):
            raise ValueError("Working directory must be workspace-relative")
        if self.expected_collected is not None and self.expected_collected < 1:
            raise ValueError("Expected collection must be positive")
        if self.pinned_environment_sha256 is not None and not re.fullmatch(
            r"[0-9a-f]{64}", self.pinned_environment_sha256):
            raise ValueError("Bad pinned environment fingerprint")

    @property
    def command_sha256(self) -> str:
        return canonical_hash(self.command)


@dataclass(frozen=True)
class VisibleManifest:
    checks: tuple[VisibleCheck, ...]
    strict: bool = False

    def __post_init__(self):
        ids = [c.check_id for c in self.checks]
        if not ids or len(ids) != len(set(ids)):
            raise ValueError("Manifest needs at least one unique selected visible check")
        if self.strict and any(c.expected_collected is None or
                               c.pinned_environment_sha256 is None or
                               not c.test_selection for c in self.checks):
            raise ValueError("Strict manifests must pin collection, environment, and selection")

    def get(self, check_id: str) -> VisibleCheck:
        for check in self.checks:
            if check.check_id == check_id:
                return check
        raise ValueError("Unselected or protected check is not allowed")

    @property
    def sha256(self) -> str:
        return canonical_hash([asdict(c) for c in self.checks])


@dataclass(frozen=True)
class TestObservation:
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    elapsed_seconds: float
    stdout: str = ""
    stderr: str = ""
    timed_out: bool = False
    setup_error: bool = False
    tests_skipped: int | None = 0

    def __post_init__(self):
        if self.elapsed_seconds < 0:
            raise ValueError("Negative duration")
        for name in ("tests_collected", "tests_passed", "tests_failed", "tests_errored", "tests_skipped"):
            v = getattr(self, name)
            if v is not None and v < 0:
                raise ValueError(f"Negative test count: {name}")

    @property
    def observed_outcome(self) -> str:
        if self.timed_out or self.setup_error or self.exit_code is None:
            return "ERROR"
        counts = (self.tests_collected, self.tests_passed,
                  self.tests_failed, self.tests_errored, self.tests_skipped)
        if any(c is None for c in counts):
            return "ERROR"
        total, passed, failed, errored, skipped = counts
        if total < 1 or passed + failed + errored + skipped != total:
            return "ERROR"
        if errored > 0:
            return "ERROR"
        if failed > 0 and self.exit_code != 0:
            return "FAIL"
        if failed > 0 and self.exit_code == 0:
            return "ERROR"  # contradictory successful process
        if self.exit_code != 0:
            return "ERROR"  # e.g. missing dependency / collection failure
        if passed < 1:
            return "ERROR"  # all skipped / no executed positive check
        return "PASS"


@dataclass(frozen=True)
class Receipt:
    receipt_id: str
    run_id: str
    task_id: str
    arm: str
    seed: int
    check_id: str
    command: tuple[str, ...]
    command_definition_sha256: str
    command_sha256: str
    working_dir: str
    test_selection: str
    expected_collected: int | None
    workspace_revision_at_start: int
    workspace_revision_at_end: int
    workspace_sha256_at_start: str
    workspace_sha256_at_end: str
    environment_sha256: str
    timestamp_start: str
    timestamp_end: str
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    tests_skipped: int | None
    qualification_error: str | None
    stdout_sha256: str
    stderr_sha256: str
    provenance: str
    observed_outcome: str
    elapsed_seconds: float
    timed_out: bool
    setup_error: bool
    test_completed_monotonic: float


@dataclass(frozen=True)
class ChangeEvent:
    action: str
    previous_revision: int
    new_revision: int
    changed: bool
    previous_sha256: str
    new_sha256: str
    edit_observed_monotonic: float


class EvidenceLedger:
    def __init__(self, root: str | Path, manifest: VisibleManifest,
                 *, run_id: str, task_id: str, arm: str, seed: int,
                 environment_sha256: str):
        if arm not in ARMS:
            raise ValueError("Unknown arm")
        for value in (run_id, task_id):
            if not ID_RE.fullmatch(value):
                raise ValueError("Unsafe run/task id")
        if not re.fullmatch(r"[0-9a-f]{64}", environment_sha256):
            raise ValueError("Expected environment SHA-256")
        self.fingerprinter = WorkspaceFingerprinter(root)
        self.manifest = manifest
        self.run_id = run_id
        self.task_id = task_id
        self.arm = arm
        self.seed = seed
        self.environment_sha256 = environment_sha256
        self.snapshot = self.fingerprinter.snapshot()
        self.revision = 0
        self.receipts: list[Receipt] = []
        self.changes: list[ChangeEvent] = []
        self.events: list[dict] = []
        self.unqualified_receipts: dict[str, str] = {}

    def observe_workspace(self, action: str) -> ChangeEvent:
        new = self.fingerprinter.snapshot()
        old = self.snapshot
        before = self.revision
        changed = new.sha256 != old.sha256
        if changed:
            self.revision += 1
            self.snapshot = new
        ev = ChangeEvent(action, before, self.revision, changed, old.sha256,
                         new.sha256, time.monotonic())
        self.changes.append(ev)
        return ev

    def set_environment(self, environment_sha256: str) -> bool:
        if not re.fullmatch(r"[0-9a-f]{64}", environment_sha256):
            raise ValueError("Expected environment SHA-256")
        changed = self.environment_sha256 != environment_sha256
        self.environment_sha256 = environment_sha256
        if changed:
            self.events.append({"type": "environment_changed", "timestamp": utc_timestamp()})
        return changed

    def status(self, receipt: Receipt) -> str:
        check = self.manifest.get(receipt.check_id)
        self.observe_workspace("status_sync")  # detect out-of-band source changes
        if self.latest(check.check_id) is not receipt:
            return "STALE"
        if (receipt.workspace_revision_at_start != receipt.workspace_revision_at_end
            or receipt.workspace_sha256_at_start != receipt.workspace_sha256_at_end
            or receipt.workspace_revision_at_end != self.revision
            or receipt.workspace_sha256_at_end != self.snapshot.sha256
            or receipt.environment_sha256 != self.environment_sha256
            or receipt.command_definition_sha256 != check.definition_sha256
            or receipt.command_sha256 != check.command_sha256
            or receipt.working_dir != check.working_dir
            or receipt.test_selection != check.test_selection
            or (check.pinned_environment_sha256 is not None and
                check.pinned_environment_sha256 != self.environment_sha256)):

            return "STALE"
        if receipt.qualification_error or receipt.receipt_id in self.unqualified_receipts:
            return "ERROR"
        return {"PASS": "CURRENT_PASS", "FAIL": "CURRENT_FAIL"}.get(
            receipt.observed_outcome, "ERROR")

    def latest(self, check_id: str) -> Receipt | None:
        self.manifest.get(check_id)
        for receipt in reversed(self.receipts):
            if receipt.check_id == check_id:
                return receipt
        return None

    def current_status(self, check_id: str) -> str:
        r = self.latest(check_id)
        return self.status(r) if r else "NOT_EXECUTED"

    def selected_current_passes(self) -> tuple[str, ...]:
        return tuple(c.check_id for c in self.manifest.checks
                     if self.current_status(c.check_id) == "CURRENT_PASS")

    def record_check(self, check_id: str, provenance: str,
                     executor: Callable[[VisibleCheck], TestObservation], *,
                     receipt_qualifier: Callable[[Receipt], None] | None = None) -> Receipt:
        if provenance not in PROVENANCES:
            raise ValueError("Invalid provenance")
        check = self.manifest.get(check_id)
        self.observe_workspace("before_visible_check")
        start = self.snapshot
        rev_start = self.revision
        started = utc_timestamp()
        outcome = executor(check)
        if not isinstance(outcome, TestObservation):
            raise TypeError("Trusted visible executor must return TestObservation")
        ended = utc_timestamp()
        self.observe_workspace("after_visible_check")
        receipt = Receipt(
            receipt_id=f"visible-{len(self.receipts)+1:06d}",
            run_id=self.run_id, task_id=self.task_id, arm=self.arm, seed=self.seed,
            check_id=check.check_id, command=check.command,
            command_definition_sha256=check.definition_sha256,
            command_sha256=check.command_sha256,
            working_dir=check.working_dir, test_selection=check.test_selection,
            expected_collected=check.expected_collected,
            workspace_revision_at_start=rev_start,
            workspace_revision_at_end=self.revision,
            workspace_sha256_at_start=start.sha256,
            workspace_sha256_at_end=self.snapshot.sha256,
            environment_sha256=self.environment_sha256,
            timestamp_start=started, timestamp_end=ended,
            exit_code=outcome.exit_code, tests_collected=outcome.tests_collected,
            tests_passed=outcome.tests_passed, tests_failed=outcome.tests_failed,
            tests_errored=outcome.tests_errored,
            tests_skipped=outcome.tests_skipped,
            qualification_error=(
                "UNEXPECTED_COLLECTION" if check.expected_collected is not None
                and outcome.tests_collected != check.expected_collected else
                "ENVIRONMENT_MISMATCH" if check.pinned_environment_sha256 is not None
                and self.environment_sha256 != check.pinned_environment_sha256 else None),
            stdout_sha256=hashlib.sha256(outcome.stdout.encode()).hexdigest(),
            stderr_sha256=hashlib.sha256(outcome.stderr.encode()).hexdigest(),
            provenance=provenance, observed_outcome=outcome.observed_outcome,
            elapsed_seconds=outcome.elapsed_seconds, timed_out=outcome.timed_out,
            setup_error=outcome.setup_error,
            test_completed_monotonic=time.monotonic())
        # A stale label is not a regression. Record an observed pass-to-fail
        # transition only when both tests completed on stable, different source
        # revisions with identical environment/command identities.
        prior_pass = next((old for old in reversed(self.receipts)
                           if old.check_id == check.check_id
                           and old.observed_outcome == "PASS"
                           and old.receipt_id not in self.unqualified_receipts
                           and old.qualification_error is None
                           and old.workspace_revision_at_start == old.workspace_revision_at_end
                           and old.workspace_sha256_at_start == old.workspace_sha256_at_end), None)
        self.receipts.append(receipt)
        # Qualify raw observations before computing derived regression events.
        if receipt_qualifier is not None:
            receipt_qualifier(receipt)
        if (prior_pass is not None
            and prior_pass.receipt_id not in self.unqualified_receipts
            and receipt.receipt_id not in self.unqualified_receipts
            and receipt.qualification_error is None
            and receipt.observed_outcome == "FAIL"
            and receipt.workspace_revision_at_start == receipt.workspace_revision_at_end
            and receipt.workspace_sha256_at_start == receipt.workspace_sha256_at_end
            and prior_pass.workspace_revision_at_end < receipt.workspace_revision_at_end
            and prior_pass.environment_sha256 == receipt.environment_sha256
            and prior_pass.command_definition_sha256 == receipt.command_definition_sha256
            and prior_pass.command_sha256 == receipt.command_sha256
            and not any(e.get("type") == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT"
                        and e.get("prior_receipt_id") == prior_pass.receipt_id
                        for e in self.events)):
            edits = [e for e in self.changes if e.changed and
                     prior_pass.workspace_revision_at_end < e.new_revision <= receipt.workspace_revision_at_end]
            invalidating_edit = edits[0] if edits else None
            self.events.append({"type": "OBSERVED_PASS_TO_FAIL_AFTER_EDIT",
                                "check_id": check.check_id,
                                "prior_receipt_id": prior_pass.receipt_id,
                                "failure_receipt_id": receipt.receipt_id,
                                "prior_revision": prior_pass.workspace_revision_at_end,
                                "failure_revision": receipt.workspace_revision_at_end,
                                "invalidating_edit_observed_monotonic": (
                                    invalidating_edit.edit_observed_monotonic if invalidating_edit else None),
                                "failure_observed_monotonic": receipt.test_completed_monotonic,
                                "edit_to_detection_seconds": (
                                    max(0.0, receipt.test_completed_monotonic - invalidating_edit.edit_observed_monotonic)
                                    if invalidating_edit else None)})
        return receipt

    def disqualify_receipt(self, receipt: Receipt, reason: str) -> None:
        """Fail closed on controller-budget violations without mutating raw observations."""
        self.unqualified_receipts[receipt.receipt_id] = reason
        self.events.append({"type": "RECEIPT_UNQUALIFIED", "receipt_id": receipt.receipt_id,
                            "reason": reason})

    def export(self) -> dict:
        # Refresh revision *before* writing any status rows or reporting a hash.
        self.observe_workspace("export_preflight")
        return {
            "schema": SCHEMA, "run_id": self.run_id, "task_id": self.task_id,
            "arm": self.arm, "seed": self.seed, "manifest_sha256": self.manifest.sha256,
            "workspace_revision": self.revision, "workspace_sha256": self.snapshot.sha256,
            "environment_sha256": self.environment_sha256,
            "receipts": [{**asdict(r), "status": self.status(r),
                          "controller_disqualification": self.unqualified_receipts.get(r.receipt_id)}
                         for r in self.receipts],
            "changes": [asdict(e) for e in self.changes],
            "events": list(self.events),
        }


@dataclass
class SharedBudget:
    max_wall_seconds: float = 900.0
    max_tool_calls: int = 60
    max_model_calls: int = 60
    used_wall_seconds: float = 0.0
    used_tool_calls: int = 0
    used_model_calls: int = 0
    used_test_runs: int = 0
    final_reserve_seconds: float = 60.0

    @property
    def remaining_wall(self) -> float:
        return max(0.0, self.max_wall_seconds - self.used_wall_seconds)

    def can_run_check(self, check: VisibleCheck, *, protect_final_reserve: bool = False) -> bool:
        reserve = self.final_reserve_seconds if protect_final_reserve else 0.0
        return (self.used_tool_calls < self.max_tool_calls
                and self.remaining_wall > 0
                and self.remaining_wall >= check.reserve_seconds + reserve)

    @property
    def remaining_repair_wall(self) -> float:
        return max(0.0, self.max_wall_seconds - self.final_reserve_seconds - self.used_wall_seconds)

    @property
    def over_limit(self) -> bool:
        return (self.used_wall_seconds > self.max_wall_seconds
                or self.used_tool_calls > self.max_tool_calls
                or self.used_model_calls > self.max_model_calls)

    def charge_model(self, elapsed_seconds: float = 0.0) -> None:
        if self.used_model_calls >= self.max_model_calls:
            raise RuntimeError("Model budget exhausted")
        self.used_model_calls += 1
        self.used_wall_seconds += elapsed_seconds

    def charge_tool(self, elapsed_seconds: float = 0.0) -> None:
        if self.used_tool_calls >= self.max_tool_calls:
            raise RuntimeError("Tool budget exhausted")
        self.used_tool_calls += 1
        self.used_wall_seconds += elapsed_seconds

    def charge_visible_test(self, elapsed_seconds: float) -> None:
        # Executing a selected check also consumes one shared tool invocation.
        self.charge_tool(elapsed_seconds)
        self.used_test_runs += 1

    def summary(self) -> dict:
        return {"used_wall_seconds": self.used_wall_seconds,
                "remaining_wall_seconds": self.remaining_wall,
                "used_tool_calls": self.used_tool_calls,
                "used_model_calls": self.used_model_calls,
                "used_test_runs": self.used_test_runs}


@dataclass(frozen=True)
class GateDecision:
    accepted: bool
    status: str
    checks: dict[str, str]
    attempts: int
    accepted_patch_sha256: str | None = None
    accepted_workspace_sha256: str | None = None
    accepted_revision: int | None = None


class VerificationController:
    """Trusted wrapper prototype. Same gate implementation for B and C.

    Executor is a trusted *visible* check adapter; NEVER pass hidden evaluator
    results. This prototype uses deterministic duration accounting. Live GP02C-03
    must enforce wall time and tool/model counters on the actual worker.
    """
    def __init__(self, ledger: EvidenceLedger, budget: SharedBudget,
                 executor: Callable[[VisibleCheck], TestObservation],
                 production_patch_sha256: Callable[[], str],
                 *, max_rechecks_per_edit: int = 4,
                 max_rechecks_per_run: int = 12):
        self.ledger = ledger
        self.budget = budget
        self.executor = executor
        self.production_patch_sha256 = production_patch_sha256
        if max_rechecks_per_edit < 0 or max_rechecks_per_run < 0:
            raise ValueError("Negative auto-recheck limit")
        self.max_rechecks_per_edit = max_rechecks_per_edit
        self.max_rechecks_per_run = max_rechecks_per_run
        self.auto_rechecks_used = 0

    def _patch_digest(self) -> str:
        digest = self.production_patch_sha256()
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Trusted production-patch capture must return a SHA-256 digest")
        return digest

    def _qualify_receipt(self, receipt: Receipt, *, check_id: str,
                         provenance: str, allowed_seconds: float) -> None:
        """Qualify budget/deadline before deriving observed-regression metrics."""
        proactive = provenance == "controller_recheck"
        if (receipt.elapsed_seconds > allowed_seconds + 1e-9 or
            self.budget.over_limit or
            (proactive and self.budget.remaining_repair_wall <= 0 and
             self.budget.used_wall_seconds > self.budget.max_wall_seconds - self.budget.final_reserve_seconds + 1e-9)):
            self.ledger.disqualify_receipt(receipt, "ACTUAL_RUNTIME_EXCEEDED_ALLOWED_DEADLINE")
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                       "check_id": check_id, "provenance": provenance,
                                       "reason": "actual_runtime_overrun_requires_live_cancellation"})

    def _run_check(self, check_id: str, provenance: str) -> Receipt | None:
        check = self.ledger.manifest.get(check_id)
        proactive = provenance == "controller_recheck"
        if not self.budget.can_run_check(check, protect_final_reserve=proactive):
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET", "check_id": check_id,
                                       "provenance": provenance})
            return None
        # CPU contract only: the caller-supplied executor is synchronous and NOT
        # cancellable. It must be replaced in GP02C-03 by a hard-deadline trusted
        # subprocess adapter, using a monotonic 840/900-second cutoff.
        allowed_seconds = (self.budget.remaining_repair_wall if proactive
                           else self.budget.remaining_wall)
        def recorded_executor(c: VisibleCheck) -> TestObservation:
            try:
                result = self.executor(c)
            except Exception as e:
                result = TestObservation(None, None, None, None, None,
                                         elapsed_seconds=0.0, setup_error=True)
                self.ledger.events.append({"type": "executor_error", "check_id": check_id,
                                           "error_type": type(e).__name__})
            self.budget.charge_visible_test(result.elapsed_seconds)
            return result
        return self.ledger.record_check(
            check_id, provenance, recorded_executor,
            receipt_qualifier=lambda receipt: self._qualify_receipt(
                receipt, check_id=check_id, provenance=provenance,
                allowed_seconds=allowed_seconds))

    def observe_agent_action(self, kind: str, action: Callable[[], object],
                             *, elapsed_seconds: float = 0.0) -> dict:
        """Wrap any tool action: read/edit/write/shell; no mandatory read gate.

        Charge the agent action itself; C post-edit checks are additional charged
        calls. Exceptions preserve before/after snapshots and are re-raised.
        """
        if (self.budget.used_tool_calls >= self.budget.max_tool_calls
            or self.budget.remaining_wall <= (
                self.budget.final_reserve_seconds if self.ledger.arm in {"B", "C"} else 0)):
            raise RuntimeError("Shared agent-tool budget exhausted or reserved for final validation")
        self.ledger.observe_workspace(f"before_{kind}")
        previously_passing = self.ledger.selected_current_passes()
        exc: BaseException | None = None
        try:
            result = action()
        except BaseException as error:
            exc = error
            result = None
        finally:
            # Conservative: one tool invocation is billed even if it errors.
            self.budget.charge_tool(elapsed_seconds)
            change = self.ledger.observe_workspace(f"after_{kind}")
        reruns: list[dict[str, str]] = []
        if change.changed and self.ledger.arm == "C":
            for position, check_id in enumerate(previously_passing):
                if (position >= self.max_rechecks_per_edit or
                    self.auto_rechecks_used >= self.max_rechecks_per_run):
                    status = "UNVERIFIED_RECHECK_LIMIT"
                    self.ledger.events.append({"type": status, "check_id": check_id})
                else:
                    self.auto_rechecks_used += 1
                    receipt = self._run_check(check_id, "controller_recheck")
                    status = ("UNVERIFIED_BUDGET" if receipt is None or
                              receipt.receipt_id in self.ledger.unqualified_receipts or
                              self.budget.over_limit else self.ledger.status(receipt))
                reruns.append({"check_id": check_id, "status": status})
        # A later recheck can mutate the worktree and stale earlier rechecks.
        # Derive the entire feedback message from one final observed state.
        self.ledger.observe_workspace("post_recheck_feedback_sync")
        for row in reruns:
            if row["status"] in {"CURRENT_PASS", "CURRENT_FAIL", "STALE", "ERROR"}:
                row["status"] = self.ledger.current_status(row["check_id"])
        feedback = {"changed": change.changed, "revision": self.ledger.revision,
                    "workspace_sha256": self.ledger.snapshot.sha256,
                    "stale_selected_checks": list(previously_passing) if change.changed else [],
                    "rechecks": reruns}
        if exc is not None:
            raise exc
        return {"result": result, "feedback": feedback}

    def agent_visible_feedback(self, result: dict) -> dict:
        """Bounded and exclusively derived from allowlisted visible-check IDs.

        Never includes test stdout, stderr, hidden tests, reference patch,
        tool outputs or the model context; the integration must preserve this.
        """
        feedback = result["feedback"]
        return {"schema": "gp02c.feedback.v1", "revision": feedback["revision"],
                "changed": feedback["changed"],
                "workspace_sha256": feedback["workspace_sha256"],
                "stale": feedback["stale_selected_checks"][:32],
                "rechecks": feedback["rechecks"][:32]}

    def final_gate(self) -> GateDecision:
        if self.ledger.arm == "A":
            raise ValueError("Ordinary baseline A has no mandatory final gate")
        # Same B/C gate. Each decision revalidates all selected statuses *after*
        # the last source sync and is bound to the workspace SHA, revision,
        # production patch SHA and frozen manifest SHA.
        self.ledger.observe_workspace("final_gate_entry")
        initial_patch = self._patch_digest()
        attempts = 0
        max_attempts = len(self.ledger.manifest.checks) * 3

        def statuses_now() -> dict[str, str]:
            self.ledger.observe_workspace("final_gate_status_sync")
            return {c.check_id: self.ledger.current_status(c.check_id)
                    for c in self.ledger.manifest.checks}

        while True:
            statuses = statuses_now()
            # Do not declare exhaustion before reviewing the final allowed result.
            if all(s == "CURRENT_PASS" for s in statuses.values()):
                self.ledger.observe_workspace("final_gate_patch_sync")
                # The final sync may itself reveal a mutation. Recompute *after* it.
                statuses = statuses_now()
                workspace = self.ledger.snapshot.sha256
                revision = self.ledger.revision
                if self._patch_digest() != initial_patch:
                    return GateDecision(False, "PATCH_CHANGED_DURING_GATE", statuses, attempts)
                # One additional consistency check after the patch capture.
                self.ledger.observe_workspace("final_gate_approval_sync")
                statuses = statuses_now()
                if (all(s == "CURRENT_PASS" for s in statuses.values()) and
                    self.ledger.snapshot.sha256 == workspace and
                    self.ledger.revision == revision and
                    self._patch_digest() == initial_patch and
                    not self.budget.over_limit and self.budget.remaining_wall > 0):
                    return GateDecision(True, "VERIFIED_VISIBLE", statuses, attempts,
                                        accepted_patch_sha256=initial_patch,
                                        accepted_workspace_sha256=workspace,
                                        accepted_revision=revision)
            if self.budget.over_limit or self.budget.remaining_wall <= 0:
                return GateDecision(False, "UNVERIFIED_BUDGET", statuses, attempts)
            if any(s in {"CURRENT_FAIL", "ERROR"} for s in statuses.values()):
                return GateDecision(False, "VISIBLE_CHECK_FAILED", statuses, attempts)
            pending = [cid for cid, s in statuses.items()
                       if s in {"STALE", "NOT_EXECUTED"}]
            if not pending:
                return GateDecision(False, "UNVERIFIED", statuses, attempts)
            if attempts >= max_attempts:
                return GateDecision(False, "UNVERIFIED_RECHECK_LIMIT", statuses, attempts)
            receipt = self._run_check(pending[0], "final_gate")
            if receipt is None or receipt.receipt_id in self.ledger.unqualified_receipts:
                return GateDecision(False, "UNVERIFIED_BUDGET", statuses_now(), attempts)
            attempts += 1


# Protected results enter ONLY after the agent and controller are finished.
# This function is an offline analyst's scorer, not an agent-visible tool.
@dataclass(frozen=True)
class AttemptRecord:
    task_id: str
    arm: str
    seed: int
    attempt_index: int
    termination: str  # SUBMITTED | TIMEOUT | STOPPED | GATE_REJECTED | INFRA_ERROR
    explicit_submit: bool
    patch_present: bool
    final_patch_sha256: str | None
    final_workspace_revision: int
    protected_result: str  # PASS | FAIL | NOT_RUN | INFRA_ERROR
    gate_decision: GateDecision | None = None
    infrastructure_retry_of: int | None = None
    final_workspace_sha256: str | None = None
    environment_sha256: str | None = None
    configuration_sha256: str | None = None
    retry_evidence_id: str | None = None

    def __post_init__(self):
        if self.arm not in ARMS or self.attempt_index < 1:
            raise ValueError("Invalid attempt identity")
        if self.protected_result not in {"PASS", "FAIL", "NOT_RUN", "INFRA_ERROR"}:
            raise ValueError("Unsupported protected grading status")
        if self.termination not in {"SUBMITTED", "TIMEOUT", "STOPPED", "GATE_REJECTED", "INFRA_ERROR"}:
            raise ValueError("Unsupported termination status")
        if self.patch_present:
            if not self.final_patch_sha256 or not re.fullmatch(r"[0-9a-f]{64}", self.final_patch_sha256):
                raise ValueError("Captured production patch needs a trusted SHA-256")
        elif self.final_patch_sha256 is not None:
            raise ValueError("Absent production patch must not have a digest")
        for name in ("final_workspace_sha256", "environment_sha256", "configuration_sha256"):
            v = getattr(self, name)
            if v is not None and not re.fullmatch(r"[0-9a-f]{64}", v):
                raise ValueError(f"Invalid {name}")


@dataclass(frozen=True)
class AttemptAssessment:
    protected_verification: str
    verified_delivery: bool
    delivery_reason: str
    termination: str
    infrastructure_fault: bool
    patch_present: bool


def assess_attempt(attempt: AttemptRecord) -> AttemptAssessment:
    """Offline-only: classify grading separately from attempted delivery.

    The trusted scorer must separately capture the final *production* patch,
    final relevant-workspace fingerprint and protected verifier result.
    """
    infra_fault = (attempt.termination == "INFRA_ERROR" or
                   attempt.protected_result == "INFRA_ERROR")
    protected = (attempt.protected_result if attempt.patch_present
                 else "INFRA_ERROR" if infra_fault else "NOT_RUN")
    def result(delivered: bool, why: str) -> AttemptAssessment:
        return AttemptAssessment(protected, delivered, why,
                                 attempt.termination, infra_fault, attempt.patch_present)
    if not attempt.patch_present:
        return result(False, "INFRASTRUCTURE_FAILURE" if infra_fault else "NO_PRODUCTION_PATCH")
    if not attempt.explicit_submit:
        return result(False, "NO_EXPLICIT_SUBMISSION")
    if attempt.termination != "SUBMITTED":
        return result(False, "NOT_DELIVERED")
    if attempt.arm in {"B", "C"}:
        gate = attempt.gate_decision
        if gate is None or not gate.accepted:
            return result(False, "FINAL_GATE_NOT_ACCEPTED")
        if (gate.accepted_patch_sha256 != attempt.final_patch_sha256 or
            gate.accepted_revision != attempt.final_workspace_revision or
            gate.accepted_workspace_sha256 != attempt.final_workspace_sha256):
            return result(False, "PATCH_OR_WORKSPACE_CHANGED_AFTER_GATE")
    if protected != "PASS":
        return result(False, "PROTECTED_VERIFICATION_NOT_PASS")
    return result(True, "VERIFIED_DELIVERY")


class AttemptRegister:
    """Keep every attempt. First attempt is primary unless *its* failure was
    independently classified as infrastructure and a documented retry exists.
    Never replace a genuine timeout or repair failure with a lucky rerun.
    """
    def __init__(self):
        self.attempts: list[AttemptRecord] = []

    def append(self, attempt: AttemptRecord) -> None:
        prior = [a for a in self.attempts if
                 (a.task_id, a.arm, a.seed) == (attempt.task_id, attempt.arm, attempt.seed)]
        if attempt.attempt_index != len(prior) + 1:
            raise ValueError("Attempt indices must be contiguous and append-only")
        if prior and attempt.infrastructure_retry_of != prior[-1].attempt_index:
            raise ValueError("Repeat must document preceding infrastructure retry")
        if prior and prior[-1].termination != "INFRA_ERROR":
            raise ValueError("Only infrastructure faults permit confirmatory replacement")
        if prior:
            previous = prior[-1]
            if not attempt.retry_evidence_id or not ID_RE.fullmatch(attempt.retry_evidence_id):
                raise ValueError("Infrastructure retry requires recorded evidence ID")
            if not all((attempt.environment_sha256, previous.environment_sha256,
                        attempt.configuration_sha256, previous.configuration_sha256)):
                raise ValueError("Infrastructure retry needs both frozen fingerprints")
            if (attempt.environment_sha256 != previous.environment_sha256 or
                attempt.configuration_sha256 != previous.configuration_sha256):
                raise ValueError("Infrastructure retry cannot change the frozen environment/config")
        self.attempts.append(attempt)

    def primary(self, task_id: str, arm: str, seed: int) -> AttemptRecord:
        matches = [a for a in self.attempts if
                   (a.task_id, a.arm, a.seed) == (task_id, arm, seed)]
        if not matches:
            raise KeyError("No attempt")
        return matches[-1]  # only documented infrastructure retries can follow


Overwriting gp02c_ledger.py


**Observation after cell 01:** `gp02c_ledger.py` should be created without exceptions. This cell only writes source code; no tests or model calls execute.

## 02 — Controller regression suite (59 tests)

Preserve the 56 GP02C-02 v1.2 controller tests and add three GP02C-03 v1.1 boundary-case probes.


In [3]:
%%writefile test_gp02c_ledger.py
"""Deterministic CPU-only contract probes. No LLM/GPU/competition task access."""
import hashlib
import json
from pathlib import Path
from tempfile import TemporaryDirectory
import unittest

from gp02c_ledger import (
    EvidenceLedger, SharedBudget, TestObservation, VerificationController,
    VisibleCheck, VisibleManifest, WorkspaceFingerprinter, environment_fingerprint,
    AttemptRecord, assess_attempt, AttemptRegister, GateDecision,
)

PASS = TestObservation(0, 1, 1, 0, 0, 0.5, stdout="1 passed")
FAIL = TestObservation(1, 1, 0, 1, 0, 0.6, stdout="1 failed")
ERROR = TestObservation(4, None, None, None, None, 0.2, setup_error=True)


class Fixtures(unittest.TestCase):
    def setUp(self):
        self.tmp = TemporaryDirectory(prefix="gp02c_02_")
        self.addCleanup(self.tmp.cleanup)
        self.root = Path(self.tmp.name)
        self.code = self.root / "app.py"
        self.code.write_text("VALUE = 1\n")
        self.env = environment_fingerprint({"python": "3.x", "wheel_sha256": "fixture-wheel-A"})
        self.check = VisibleCheck("x", ("pytest", "-q", "tests/test_x.py"),
                                  hashlib.sha256(b"visible-check-definition-x").hexdigest(), 0.4,
                                  working_dir=".", test_selection="tests/test_x.py",
                                  expected_collected=1, pinned_environment_sha256=self.env)
        self.check2 = VisibleCheck("y", ("pytest", "-q", "tests/test_y.py"),
                                   hashlib.sha256(b"visible-check-definition-y").hexdigest(), 0.4,
                                   working_dir=".", test_selection="tests/test_y.py",
                                   expected_collected=1, pinned_environment_sha256=self.env)
        self.manifest = VisibleManifest((self.check,), strict=True)

    def ledger(self, arm="C", manifest=None):
        return EvidenceLedger(self.root, manifest or self.manifest, run_id="fixture-001",
                              task_id="cpu_fixture", arm=arm, seed=20261105,
                              environment_sha256=self.env)

    def ctrl(self, arm="C", budget=None, executor=None, manifest=None):
        ledger = self.ledger(arm, manifest)
        return VerificationController(ledger, budget or SharedBudget(),
                                      executor or (lambda c: PASS),
                                      production_patch_sha256=lambda: hashlib.sha256(self.code.read_bytes()).hexdigest())

    def test_01_pass_then_edit_stales_without_regression(self):
        led = self.ledger()
        receipt = led.record_check("x", "agent", lambda _: PASS)
        self.assertEqual(led.status(receipt), "CURRENT_PASS")
        self.code.write_text("VALUE = 2\n")
        ev = led.observe_workspace("edit_file")
        self.assertTrue(ev.changed)
        self.assertEqual(ev.new_revision, 1)
        self.assertEqual(led.status(receipt), "STALE")
        self.assertEqual(receipt.observed_outcome, "PASS")

    def test_02_post_edit_pass_new_receipt(self):
        led = self.ledger()
        old = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 2\n")
        new = led.record_check("x", "controller_recheck", lambda _: PASS)
        self.assertEqual(led.status(old), "STALE")
        self.assertEqual(led.status(new), "CURRENT_PASS")
        self.assertNotEqual(old.workspace_revision_at_end, new.workspace_revision_at_end)

    def test_03_pass_then_edit_then_fail_observed_regression(self):
        ctrl = self.ctrl()
        led = ctrl.ledger
        led.record_check("x", "agent", lambda _: PASS)
        result = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 2\n"))
        self.assertEqual(result["feedback"]["rechecks"], [{"check_id": "x", "status": "CURRENT_PASS"}])
        self.assertEqual(ctrl.budget.used_test_runs, 1)
        self.assertEqual(led.current_status("x"), "CURRENT_PASS")
        # Separate controller recheck explicitly fails when regression is visible.
        ctrl2 = self.ctrl(executor=lambda c: FAIL)
        ctrl2.ledger.record_check("x", "agent", lambda _: PASS)
        out = ctrl2.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 3\n"))
        self.assertEqual(out["feedback"]["rechecks"][0]["status"], "CURRENT_FAIL")
        self.assertEqual(ctrl2.ledger.latest("x").observed_outcome, "FAIL")
        self.assertTrue(any(e["type"] == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT"
                            for e in ctrl2.ledger.events))

    def test_04_rejected_or_noop_edit_does_not_advance_revision(self):
        ctrl = self.ctrl()
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        def rejected():
            raise PermissionError("restricted")
        with self.assertRaises(PermissionError):
            ctrl.observe_agent_action("edit_file", rejected)
        self.assertEqual(ctrl.ledger.revision, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_PASS")
        self.assertEqual(ctrl.budget.used_tool_calls, 1)

    def test_05_shell_mutation_detected(self):
        led = self.ledger()
        led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 5\n")
        self.assertEqual(led.current_status("x"), "STALE")
        self.assertEqual(led.revision, 1)

    def test_06_test_itself_mutates_code_invalid_receipt(self):
        led = self.ledger()
        def mutating(_):
            self.code.write_text("VALUE = 6\n")
            return PASS
        receipt = led.record_check("x", "agent", mutating)
        self.assertEqual(led.status(receipt), "STALE")
        self.assertEqual(receipt.observed_outcome, "PASS")

    def test_07_zero_collection_is_not_pass(self):
        led = self.ledger()
        receipt = led.record_check("x", "agent", lambda _: TestObservation(0, 0, 0, 0, 0, 0.1))
        self.assertEqual(led.status(receipt), "ERROR")

    def test_08_timeout_error_malformed_results_and_skipped(self):
        samples = [
            TestObservation(None, None, None, None, None, 0.4, timed_out=True),
            ERROR,
            TestObservation(0, None, None, None, None, 0.2),
            TestObservation(0, 1, 0, 0, 0, 0.1, tests_skipped=1),
            TestObservation(0, 1, 0, 1, 0, 0.1),
        ]
        for sample in samples:
            with self.subTest(observation=sample):
                led = self.ledger()
                r = led.record_check("x", "agent", lambda _: sample)
                self.assertEqual(led.status(r), "ERROR")

    def test_09_environment_change_invalidates_but_not_workspace_revision(self):
        led = self.ledger()
        r = led.record_check("x", "agent", lambda _: PASS)
        led.set_environment(environment_fingerprint({"python": "3.x", "wheel_sha256": "B"}))
        self.assertEqual(led.status(r), "STALE")
        self.assertEqual(led.revision, 0)

    def test_10_revert_bytes_cannot_resurrect_old_pass(self):
        led = self.ledger()
        r = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 9\n")
        led.observe_workspace("modify")
        self.code.write_text("VALUE = 1\n")
        led.observe_workspace("revert")
        self.assertEqual(led.revision, 2)
        self.assertEqual(led.snapshot.sha256, r.workspace_sha256_at_end)
        self.assertEqual(led.status(r), "STALE")

    def test_11_no_prior_pass_no_auto_recheck(self):
        ctrl = self.ctrl()
        out = ctrl.observe_agent_action("write_file", lambda: self.code.write_text("VALUE = 11\n"))
        self.assertEqual(out["feedback"]["rechecks"], [])
        self.assertEqual(ctrl.budget.used_tool_calls, 1)
        self.assertEqual(ctrl.budget.used_test_runs, 0)

    def test_12_C_insufficient_budget_does_not_fake_pass(self):
        budget = SharedBudget(max_wall_seconds=0.5, max_tool_calls=3, final_reserve_seconds=0.0)
        ctrl = self.ctrl(budget=budget)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        out = ctrl.observe_agent_action("write_file", lambda: self.code.write_text("VALUE = 12\n"),
                                        elapsed_seconds=0.2)
        self.assertEqual(out["feedback"]["rechecks"],
                         [{"check_id": "x", "status": "UNVERIFIED_BUDGET"}])
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "STALE")

    def test_13_B_final_gate_rechecks_after_edit(self):
        ctrl = self.ctrl(arm="B")
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 13\n")
        decision = ctrl.final_gate()
        self.assertTrue(decision.accepted)
        self.assertEqual(decision.status, "VERIFIED_VISIBLE")
        self.assertEqual(ctrl.ledger.latest("x").provenance, "final_gate")
        self.assertEqual(ctrl.budget.used_tool_calls, 1)

    def test_14_B_final_gate_fails_actual_failure(self):
        ctrl = self.ctrl(arm="B", executor=lambda _: FAIL)
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(dec.status, "VISIBLE_CHECK_FAILED")
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_FAIL")

    def test_15_B_C_identical_final_gate(self):
        for arm in ("B", "C"):
            with self.subTest(arm=arm):
                ctrl = self.ctrl(arm=arm)
                dec = ctrl.final_gate()
                self.assertTrue(dec.accepted)
                self.assertEqual(dec.attempts, 1)
                self.assertEqual(ctrl.budget.used_test_runs, 1)

    def test_16_A_no_final_gate(self):
        ctrl = self.ctrl(arm="A")
        with self.assertRaises(ValueError):
            ctrl.final_gate()

    def test_17_protected_or_unlisted_check_blocked(self):
        with self.assertRaises(ValueError):
            VisibleCheck("hidden", ("pytest", "hidden.py"), "0" * 64,
                         origin="protected_evaluator")
        with self.assertRaises(ValueError):
            self.ledger().record_check("hidden", "agent", lambda _: PASS)

    def test_18_feedback_never_contains_private_outputs(self):
        secret = "PROTECTED_REFERENCE_PATCH: DO_NOT_EXPOSE"
        ctrl = self.ctrl(executor=lambda c: TestObservation(0, 1, 1, 0, 0, 0.3,
                                                             stdout=secret, stderr=secret))
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        result = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 18\n"))
        feedback = json.dumps(ctrl.agent_visible_feedback(result))
        self.assertNotIn(secret, feedback)
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_PASS")

    def test_19_no_mandatory_investigation_tool_call(self):
        for arm in ("A", "B", "C"):
            with self.subTest(arm=arm):
                ctrl = self.ctrl(arm=arm)
                outcome = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 19\n"))
                self.assertTrue(outcome["feedback"]["changed"] or ctrl.ledger.revision == 0)
                self.assertEqual(ctrl.budget.used_tool_calls, 1)

    def test_20_untracked_addition_deletion_and_exclusion(self):
        fp = WorkspaceFingerprinter(self.root)
        first = fp.snapshot()
        untracked = self.root / "helper.py"
        untracked.write_text("print('helper')\n")
        second = fp.snapshot()
        self.assertNotEqual(first.sha256, second.sha256)
        untracked.unlink()
        self.assertEqual(fp.snapshot().sha256, first.sha256)
        cache = self.root / "__pycache__"
        cache.mkdir()
        (cache / "app.pyc").write_bytes(b"ignored")
        self.assertEqual(fp.snapshot().sha256, first.sha256)

    def test_21_failure_supersedes_earlier_pass_same_revision(self):
        led = self.ledger()
        old = led.record_check("x", "agent", lambda _: PASS)
        new = led.record_check("x", "agent", lambda _: FAIL)
        self.assertEqual(led.status(old), "STALE")
        self.assertEqual(led.status(new), "CURRENT_FAIL")

    def test_22_B_final_budget_exhaustion(self):
        ctrl = self.ctrl(arm="B", budget=SharedBudget(max_wall_seconds=0.2))
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(dec.status, "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.budget.used_test_runs, 0)

    def test_23_test_mutation_prevents_final_acceptance(self):
        def mutate(check):
            self.code.write_text(self.code.read_text() + "# touched\n")
            return PASS
        ctrl = self.ctrl(arm="B", executor=mutate)
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertIn(dec.status, {"UNVERIFIED_RECHECK_LIMIT", "UNVERIFIED_BUDGET"})

    def test_24_agent_action_C_auto_rechecks_and_charges_shared_calls(self):
        ctrl = self.ctrl()
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.observe_agent_action("run_command", lambda: self.code.write_text("VALUE = 24\n"),
                                  elapsed_seconds=2.0)
        self.assertEqual(ctrl.budget.used_tool_calls, 2)
        self.assertEqual(ctrl.budget.used_test_runs, 1)
        self.assertAlmostEqual(ctrl.budget.used_wall_seconds, 2.5)

    def test_25_B_does_not_proactively_recheck(self):
        ctrl = self.ctrl(arm="B")
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 25\n"))
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "STALE")

    def test_26_multiple_selected_checks_C_proactive_scope(self):
        manifest = VisibleManifest((self.check, self.check2))
        ctrl = self.ctrl(manifest=manifest)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.ledger.record_check("y", "agent", lambda _: FAIL)
        res = ctrl.observe_agent_action("write_file", lambda: self.code.write_text("VALUE = 26\n"))
        self.assertEqual([x["check_id"] for x in res["feedback"]["rechecks"]], ["x"])
        self.assertEqual(ctrl.ledger.current_status("y"), "STALE")

    def test_27_executor_exception_does_not_approve(self):
        def explode(_):
            raise FileNotFoundError("Missing test setup")
        ctrl = self.ctrl(arm="B", executor=explode)
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(ctrl.ledger.current_status("x"), "ERROR")

    def test_29_test_actual_runtime_overrun_cannot_be_accepted(self):
        ctrl = self.ctrl(arm="B", budget=SharedBudget(max_wall_seconds=1.0),
                         executor=lambda c: TestObservation(0, 1, 1, 0, 0, 3.0))
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(dec.status, "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.budget.used_test_runs, 1)

    def test_30_prevent_action_after_tool_budget_consumed(self):
        ctrl = self.ctrl(budget=SharedBudget(max_tool_calls=0))
        before = self.code.read_text()
        with self.assertRaises(RuntimeError):
            ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("illegal"))
        self.assertEqual(self.code.read_text(), before)

    def test_28_observation_outcome_and_status_separate(self):
        led = self.ledger()
        r = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 28\n")
        exported = led.export()
        self.assertEqual(exported["receipts"][0]["observed_outcome"], "PASS")
        self.assertEqual(exported["receipts"][0]["status"], "STALE")
        self.assertNotIn("PROTECTED_REFERENCE_PATCH", json.dumps(exported))

    def test_31_strict_manifest_requires_test_selection_collection_and_environment(self):
        incomplete = VisibleCheck("x", ("pytest", "-q", "tests/x.py"), "0" * 64)
        with self.assertRaises(ValueError):
            VisibleManifest((incomplete,), strict=True)
        self.assertTrue(VisibleManifest((self.check,), strict=True).strict)
        with self.assertRaises(ValueError):
            VisibleCheck("z", ("pytest", "-q"), "0"*64, working_dir="../outside")

    def test_32_expected_collection_mismatch_not_an_accepted_pass(self):
        led = self.ledger()
        obs = TestObservation(0, 2, 2, 0, 0, 0.1)
        receipt = led.record_check("x", "agent", lambda _: obs)
        self.assertEqual(receipt.observed_outcome, "PASS")
        self.assertEqual(receipt.qualification_error, "UNEXPECTED_COLLECTION")
        self.assertEqual(led.status(receipt), "ERROR")
        ctrl = self.ctrl(arm="B", executor=lambda _: obs)
        self.assertFalse(ctrl.final_gate().accepted)

    def test_33_collection_count_consistency_and_skipped(self):
        samples = [
            TestObservation(0, 1, 1, 0, 0, 0.1, tests_skipped=1),
            TestObservation(0, 2, 0, 0, 0, 0.1, tests_skipped=2),
            TestObservation(0, 2, 1, 0, 0, 0.1, tests_skipped=1),
        ]
        self.assertEqual([x.observed_outcome for x in samples], ["ERROR", "ERROR", "PASS"])

    def test_34_final_gate_patch_binding_and_post_gate_edit_rejection(self):
        ctrl = self.ctrl(arm="B")
        dec = ctrl.final_gate()
        self.assertTrue(dec.accepted)
        old_digest = dec.accepted_patch_sha256
        self.assertEqual(old_digest, hashlib.sha256(self.code.read_bytes()).hexdigest())
        self.code.write_text("VALUE = 99\n")
        self.assertNotEqual(old_digest, hashlib.sha256(self.code.read_bytes()).hexdigest())
        assessed = assess_attempt(AttemptRecord(
            "task", "B", 20261105, 1, "SUBMITTED", True, True,
            hashlib.sha256(self.code.read_bytes()).hexdigest(),
            dec.accepted_revision + 1, "PASS", dec))
        self.assertEqual(assessed.protected_verification, "PASS")
        self.assertFalse(assessed.verified_delivery)
        self.assertEqual(assessed.delivery_reason, "PATCH_OR_WORKSPACE_CHANGED_AFTER_GATE")

    def test_35_controller_test_mutates_production_patch_during_gate(self):
        ctrl = self.ctrl(arm="B", executor=lambda _: (
            self.code.write_text("VALUE = 42\n"), PASS)[1])
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)

    def test_36_C_rechecks_bounded_and_manifest_ordered(self):
        manifest = VisibleManifest((self.check, self.check2), strict=True)
        ctrl = self.ctrl(manifest=manifest)
        ctrl.max_rechecks_per_edit = 1
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.ledger.record_check("y", "agent", lambda _: PASS)
        out = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 7\n"))
        self.assertEqual(out['feedback']['rechecks'], [
            {"check_id": "x", "status": "CURRENT_PASS"},
            {"check_id": "y", "status": "UNVERIFIED_RECHECK_LIMIT"}])
        self.assertEqual(ctrl.ledger.current_status("y"), "STALE")
        self.assertEqual(ctrl.auto_rechecks_used, 1)

    def test_37_C_preserves_final_validation_reserve(self):
        budget = SharedBudget(max_wall_seconds=65, used_wall_seconds=5, final_reserve_seconds=60)
        ctrl = self.ctrl(budget=budget)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        with self.assertRaises(RuntimeError):
            ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("NOT ALLOWED\n"))
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(self.code.read_text(), "VALUE = 1\n")

    def test_38_C_proactive_recheck_cannot_spend_final_reserve(self):
        budget = SharedBudget(max_wall_seconds=65, used_wall_seconds=4.8, final_reserve_seconds=60)
        ctrl = self.ctrl(budget=budget)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        result = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 88\n"))
        self.assertEqual(result["feedback"]["rechecks"][0]["status"], "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "STALE")

    def test_39_protected_grade_separate_from_verified_delivery(self):
        digest = "f" * 64
        for term in ("TIMEOUT", "STOPPED", "GATE_REJECTED"):
            rec = AttemptRecord("task", "A", 20261105, 1, term,
                                False, True, digest, 1, "PASS")
            outcome = assess_attempt(rec)
            self.assertEqual(outcome.protected_verification, "PASS")
            self.assertFalse(outcome.verified_delivery)
        successful = AttemptRecord("task", "A", 20261105, 1,
                                   "SUBMITTED", True, True, digest, 1, "PASS")
        self.assertTrue(assess_attempt(successful).verified_delivery)

    def test_40_rejected_gate_can_still_have_protected_pass(self):
        rec = AttemptRecord("task", "C", 20261105, 1, "GATE_REJECTED", True,
                            True, "c"*64, 2, "PASS", GateDecision(False,"VISIBLE_CHECK_FAILED",{},1))
        result = assess_attempt(rec)
        self.assertEqual(result.protected_verification, "PASS")
        self.assertFalse(result.verified_delivery)
        self.assertEqual(result.delivery_reason, "NOT_DELIVERED")

    def test_41_no_patch_or_unexecuted_protected_check(self):
        rec = AttemptRecord("task", "B", 20261105, 1, "TIMEOUT", False,
                            False, None, 0, "NOT_RUN")
        res = assess_attempt(rec)
        self.assertEqual(res.protected_verification, "NOT_RUN")
        self.assertFalse(res.verified_delivery)

    def test_42_successful_gate_bound_to_exact_production_patch(self):
        ctrl = self.ctrl(arm="C")
        gate = ctrl.final_gate()
        digest = hashlib.sha256(self.code.read_bytes()).hexdigest()
        rec = AttemptRecord("task", "C", 20261105, 1, "SUBMITTED", True,
                            True, digest, gate.accepted_revision, "PASS", gate,
                            final_workspace_sha256=gate.accepted_workspace_sha256)
        self.assertTrue(assess_attempt(rec).verified_delivery)

    def test_43_attempt_registry_preserves_failures_without_replacing(self):
        reg = AttemptRegister()
        first = AttemptRecord("task", "B", 1, 1, "TIMEOUT", False, True,
                              "a"*64, 0, "FAIL")
        reg.append(first)
        second = AttemptRecord("task", "B", 1, 2, "SUBMITTED", True, True,
                               "b"*64, 1, "PASS")
        with self.assertRaises(ValueError):
            reg.append(second)
        self.assertEqual(reg.primary("task", "B", 1), first)

    def test_44_infra_retry_preserved_and_explicitly_selected(self):
        reg = AttemptRegister()
        first = AttemptRecord("task", "B", 1, 1, "INFRA_ERROR", False,
                              False, None, 0, "INFRA_ERROR",
                              environment_sha256="e"*64, configuration_sha256="c"*64)
        second = AttemptRecord("task", "B", 1, 2, "SUBMITTED", True,
                               True, "d"*64, 1, "PASS", infrastructure_retry_of=1,
                               environment_sha256="e"*64, configuration_sha256="c"*64,
                               retry_evidence_id="infra-case-001")
        reg.append(first)
        reg.append(second)
        self.assertEqual(reg.primary("task", "B", 1), second)
        self.assertEqual(len(reg.attempts), 2)

    def test_45_observed_regression_clock_from_first_invalidating_edit(self):
        ctrl = self.ctrl(executor=lambda _: FAIL)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 15\n"))
        ev = [x for x in ctrl.ledger.events if x.get('type') == 'OBSERVED_PASS_TO_FAIL_AFTER_EDIT'][0]
        self.assertLessEqual(ev['invalidating_edit_observed_monotonic'],
                             ev['failure_observed_monotonic'])
        self.assertGreaterEqual(ev['edit_to_detection_seconds'], 0)

    def test_46_immutability_of_recorded_pass_outcome(self):
        led = self.ledger()
        previous = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 6\n")
        led.observe_workspace("edit_file")
        self.assertEqual(previous.observed_outcome,"PASS")
        self.assertEqual(led.status(previous),"STALE")

    def test_47_gate_rechecks_after_late_workspace_mutation_without_patch_change(self):
        ctrl = self.ctrl(arm="B")
        old = ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        cfg = self.root / "runtime.cfg"
        cfg.write_text("A\n")
        # Re-run current as the new config itself made the original receipt stale.
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        original = ctrl.ledger.observe_workspace
        injected = [False]
        def late(action):
            if action == "final_gate_patch_sync" and not injected[0]:
                cfg.write_text("B\n")
                injected[0] = True
            return original(action)
        ctrl.ledger.observe_workspace = late
        decision = ctrl.final_gate()
        self.assertTrue(injected[0]); self.assertTrue(decision.accepted)
        self.assertEqual(decision.attempts, 1)
        self.assertNotEqual(old.workspace_sha256_at_end, decision.accepted_workspace_sha256)
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_PASS")
        self.assertEqual(decision.accepted_workspace_sha256, ctrl.ledger.snapshot.sha256)

    def test_48_gate_final_allowed_recheck_success_is_accepted(self):
        count = [0]
        config = self.root / "runtime.cfg"
        def run(_):
            count[0] += 1
            if count[0] < 3:
                config.write_text(str(count[0]))
            return PASS
        ctrl = self.ctrl(arm="B", executor=run)
        decision = ctrl.final_gate()
        self.assertEqual(count[0], 3)
        self.assertTrue(decision.accepted)
        self.assertEqual(decision.attempts, 3)

    def test_49_C_overrunning_proactive_test_cannot_qualify(self):
        budget = SharedBudget(max_wall_seconds=900, used_wall_seconds=839.3,
                              final_reserve_seconds=60)
        ctrl = self.ctrl(budget=budget, executor=lambda _: TestObservation(0,1,1,0,0,1.3))
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        res = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 49\n"))
        self.assertEqual(res["feedback"]["rechecks"][0]["status"], "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.ledger.current_status("x"), "ERROR")
        self.assertGreater(budget.used_wall_seconds, 840.0) # honest elapsed accounting
        self.assertTrue(any(e.get("reason")=="ACTUAL_RUNTIME_EXCEEDED_ALLOWED_DEADLINE"
                            for e in ctrl.ledger.events))

    def test_57_C_batch_feedback_reflects_final_workspace_state(self):
        manifest = VisibleManifest((self.check, self.check2), strict=True)
        cfg = self.root / "runtime.cfg"
        def run(c):
            if c.check_id == "y":
                cfg.write_text("config changed during second check\n")
            return PASS
        ctrl = self.ctrl(manifest=manifest, executor=run)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.ledger.record_check("y", "agent", lambda _: PASS)
        out = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 57\n"))
        self.assertEqual([r["status"] for r in out["feedback"]["rechecks"]],
                         ["STALE", "STALE"])
        self.assertEqual(out["feedback"]["revision"], ctrl.ledger.revision)
        self.assertEqual(out["feedback"]["workspace_sha256"], ctrl.ledger.snapshot.sha256)
        self.assertEqual(ctrl.agent_visible_feedback(out)["rechecks"][0]["status"], "STALE")

    def test_58_unqualified_fail_must_not_be_observed_regression(self):
        budget = SharedBudget(max_wall_seconds=900, used_wall_seconds=839.2,
                              final_reserve_seconds=60)
        fail_slow = TestObservation(1, 1, 0, 1, 0, 1.5)
        ctrl = self.ctrl(budget=budget, executor=lambda _: fail_slow)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        out = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 58\n"))
        self.assertEqual(out["feedback"]["rechecks"][0]["status"], "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.ledger.receipts[-1].observed_outcome, "FAIL")
        self.assertIn(ctrl.ledger.receipts[-1].receipt_id, ctrl.ledger.unqualified_receipts)
        self.assertFalse(any(e.get("type") == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT"
                             for e in ctrl.ledger.events))

    def test_59_unqualified_prior_pass_not_regression_origin(self):
        ctrl = self.ctrl(executor=lambda _: FAIL)
        old = ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.ledger.disqualify_receipt(old, "INTENTIONAL_PROBE_DISQUALIFICATION")
        ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 59\n"))
        self.assertFalse(any(e.get("type") == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT"
                             for e in ctrl.ledger.events))

    def test_50_no_patch_infra_failure_retains_classification(self):
        attempt = AttemptRecord("task", "B", 1, 1, "INFRA_ERROR", False,
                                False, None, 0, "INFRA_ERROR")
        assessment = assess_attempt(attempt)
        self.assertEqual(assessment.protected_verification, "INFRA_ERROR")
        self.assertTrue(assessment.infrastructure_fault)
        self.assertEqual(assessment.termination, "INFRA_ERROR")
        self.assertFalse(assessment.patch_present)

    def test_51_same_patch_digest_different_workspace_fails_delivery(self):
        ctrl = self.ctrl(arm="B")
        gate = ctrl.final_gate(); self.assertTrue(gate.accepted)
        cfg = self.root / "runtime.cfg"; cfg.write_text("B\n")
        ctrl.ledger.observe_workspace("out_of_band_edit")
        rec = AttemptRecord("task", "B", 1, 1, "SUBMITTED", True,
                            True, gate.accepted_patch_sha256, ctrl.ledger.revision,
                            "PASS", gate, final_workspace_sha256=ctrl.ledger.snapshot.sha256)
        assessment = assess_attempt(rec)
        self.assertFalse(assessment.verified_delivery)
        self.assertEqual(assessment.protected_verification, "PASS")
        self.assertEqual(assessment.delivery_reason, "PATCH_OR_WORKSPACE_CHANGED_AFTER_GATE")

    def test_52_missing_delivery_workspace_is_never_accepted(self):
        ctrl = self.ctrl(arm="B")
        gate = ctrl.final_gate()
        rec = AttemptRecord("task", "B", 1, 1, "SUBMITTED", True,
                            True, gate.accepted_patch_sha256, gate.accepted_revision,
                            "PASS", gate)
        self.assertFalse(assess_attempt(rec).verified_delivery)

    def test_53_retry_requires_same_config_env_and_evidence(self):
        first = AttemptRecord("task", "B", 1, 1, "INFRA_ERROR", False,
                              False, None, 0, "INFRA_ERROR",
                              environment_sha256="e"*64, configuration_sha256="c"*64)
        reg = AttemptRegister();reg.append(first)
        for env, cfg, evidence in (("e"*64,"d"*64,"case"),("f"*64,"c"*64,"case"),("e"*64,"c"*64,None)):
            with self.subTest(env=env, cfg=cfg, evidence=evidence):
                second = AttemptRecord("task", "B", 1, 2, "SUBMITTED", True,
                                       True, "b"*64, 1, "PASS", infrastructure_retry_of=1,
                                       environment_sha256=env, configuration_sha256=cfg,
                                       retry_evidence_id=evidence)
                with self.assertRaises(ValueError):reg.append(second)
        self.assertEqual(len(reg.attempts), 1)

    def test_54_last_edit_after_gate_stales_receipt_and_export_refreshes(self):
        ctrl = self.ctrl(arm="B")
        gate = ctrl.final_gate();self.assertTrue(gate.accepted)
        self.code.write_text("VALUE = 54\n")
        export = ctrl.ledger.export()
        self.assertNotEqual(export["workspace_sha256"],gate.accepted_workspace_sha256)
        self.assertEqual(export["receipts"][-1]["status"],"STALE")

    def test_55_B_C_identical_new_final_gate_after_workflow_change(self):
        for arm in ("B", "C"):
            with self.subTest(arm=arm):
                ctrl = self.ctrl(arm=arm)
                old = ctrl.ledger.record_check("x", "agent", lambda _: PASS)
                self.code.write_text(f"VALUE = {arm!r}\n")
                gate = ctrl.final_gate()
                self.assertTrue(gate.accepted)
                self.assertEqual(gate.attempts,1)
                self.assertEqual(ctrl.ledger.status(old),"STALE")

    def test_56_visible_budget_overrun_cannot_be_final_approval(self):
        budget = SharedBudget(max_wall_seconds=1.0, final_reserve_seconds=0)
        ctrl = self.ctrl(arm="B", budget=budget, executor=lambda _: TestObservation(0,1,1,0,0,1.1))
        decision = ctrl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual(decision.status,"UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.ledger.current_status("x"),"ERROR")


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_gp02c_ledger.py


**Observation after cell 02:** `test_gp02c_ledger.py` should be created without exceptions. This cell only writes source code; no tests or model calls execute.

## 03 — Implement trusted real-`pytest` adapter

Run selected tests on temporary separate source copies; emit JUnit and attestation; enforce hard 840/900 cutoffs via cancellable process groups; never execute protected tests.

In [4]:
%%writefile gp02c_live_adapter.py
"""GP02C-03: CPU-only trusted visible-pytest runner and B/C integration bridge.

No model, ADK harness, protected evaluator or competition task is invoked here.
The source tree is copied to a separate best-effort read-only snapshot per test.
This protects the live working tree from accidental test mutations; it is NOT a
security sandbox for adversarial code (no container/mount namespace isolation).
"""
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Mapping
import hashlib
import json
import os
import platform
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import xml.etree.ElementTree as ET

from gp02c_ledger import (
    EvidenceLedger, GateDecision, SharedBudget, TestObservation, VerificationController,
    VisibleCheck, WorkspaceFingerprinter, canonical_hash, environment_fingerprint,
)

SCHEMA = "gp02c.live-adapter.v1.0"


def utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(262144), b""):
            h.update(block)
    return h.hexdigest()


def frozen_pytest_environment(extra: Mapping[str, str] | None = None) -> dict[str, str]:
    """The CPU fixture profile; production profiles must pin dependencies separately."""
    env = os.environ.copy()
    env.pop("PYTEST_ADDOPTS", None)
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONHASHSEED"] = "0"
    if extra:
        for k, v in extra.items():
            if not isinstance(k, str) or not isinstance(v, str):
                raise ValueError("Environment overrides must be strings")
            env[k] = v
    return env


def observed_environment_identity(extra: Mapping[str, str] | None = None) -> tuple[str, dict]:
    """For CPU smoke only; a production run must hash pinned installed wheels."""
    import pytest
    profile = {
        "python_version": platform.python_version(),
        "python_executable": str(Path(sys.executable).resolve()),
        "python_executable_sha256": sha256_file(Path(sys.executable).resolve()),
        "pytest_version": pytest.__version__,
        "platform": platform.platform(),
        "pytest_plugin_autoload": "disabled",
        "python_write_bytecode": "disabled",
        "extra": dict(sorted((extra or {}).items())),
    }
    return environment_fingerprint(profile), profile


@dataclass
class RealDeadline:
    """One absolute clock per attempt. 840/900s defaults; no reset per check."""
    repair_cutoff_seconds: float = 840.0
    overall_cutoff_seconds: float = 900.0
    started_monotonic: float | None = None
    now: Callable[[], float] = time.monotonic

    def __post_init__(self):
        if not 0 < self.repair_cutoff_seconds < self.overall_cutoff_seconds:
            raise ValueError("Need 0 < repair cutoff < overall cutoff")
        if self.started_monotonic is None:
            self.started_monotonic = self.now()

    def remaining(self, phase: str) -> float:
        if phase not in {"ordinary", "proactive", "final"}:
            raise ValueError("Unrecognized execution phase")
        cap = self.overall_cutoff_seconds if phase == "final" else self.repair_cutoff_seconds
        return max(0.0, self.started_monotonic + cap - self.now())

    def elapsed(self) -> float:
        return self.now() - self.started_monotonic


@dataclass(frozen=True)
class PytestResult:
    observation: TestObservation
    junit_sha256: str | None
    source_sha256_before: str
    source_sha256_after: str
    copied_snapshot_sha256: str | None
    snapshot_sha256_after_test: str | None
    execution_dir: str
    process_cancelled: bool
    stdout_file_sha256: str | None
    stderr_file_sha256: str | None
    raw_exit_code: int | None


def junit_counts(junit_path: Path) -> tuple[int, int, int, int, int]:
    """Extract actual testcase elements; never trust exit code or suite labels alone."""
    data = junit_path.read_bytes()
    root = ET.fromstring(data)
    cases = [el for el in root.iter() if el.tag.rsplit("}", 1)[-1] == "testcase"]
    collected = len(cases)
    fail = sum(any(c.tag.rsplit("}", 1)[-1] == "failure" for c in t) for t in cases)
    err = sum(any(c.tag.rsplit("}", 1)[-1] == "error" for c in t) for t in cases)
    skipped = sum(any(c.tag.rsplit("}", 1)[-1] == "skipped" for c in t) for t in cases)
    passed = collected - fail - err - skipped
    if passed < 0 or collected != fail + err + skipped + passed:
        raise ValueError("Contradictory JUnit testcase counts")
    return collected, passed, fail, err, skipped


def _read_head(path: Path, max_bytes: int = 6000) -> str:
    with path.open("rb") as f:
        return f.read(max_bytes).decode("utf-8", errors="replace")


def _cancel_process_group(proc: subprocess.Popen, grace: float = 0.15) -> None:
    if proc.poll() is not None:
        return
    if os.name == "posix":
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            return
    else:
        proc.terminate()
    try:
        proc.wait(timeout=grace)
    except subprocess.TimeoutExpired:
        if os.name == "posix":
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
        else:
            proc.kill()
        proc.wait(timeout=2)


def cancelable_argv(argv: tuple[str, ...], cwd: Path, env: dict[str, str],
                    stdout_path: Path, stderr_path: Path, timeout: float) -> tuple[int | None, bool, float]:
    """Never shell=True. Strict real subprocess timeout, terminate process group."""
    if timeout <= 0:
        return None, True, 0.0
    start = time.monotonic()
    with stdout_path.open("wb") as out, stderr_path.open("wb") as err:
        creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0
        proc = subprocess.Popen(list(argv), cwd=str(cwd), env=env, stdout=out, stderr=err,
                                start_new_session=(os.name == "posix"), creationflags=creationflags)
        try:
            proc.wait(timeout=timeout)
            timed_out = False
        except subprocess.TimeoutExpired:
            timed_out = True
            _cancel_process_group(proc)
    return proc.returncode, timed_out, time.monotonic() - start


def copy_isolated_snapshot(src_root: Path, dest_root: Path) -> str:
    """Copy the exact fingerprinter inclusion set without following symlinks.

    This is non-adversarial isolation; read-only chmod is best-effort. Any
    mutation is detected and disqualifies the check. Do not run hostile code.
    """
    fp = WorkspaceFingerprinter(src_root)
    before = fp.snapshot()
    dest_root.mkdir(exist_ok=False)
    for relative, kind, _digest in before.entries:
        if kind != "F":
            raise ValueError("Symlinks require a separately validated snapshot policy")
        source = src_root / relative
        dest = dest_root / relative
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, dest, follow_symlinks=False)
        dest.chmod(0o444)
    snapshot_hash = WorkspaceFingerprinter(dest_root).snapshot().sha256
    if before.sha256 != snapshot_hash or fp.snapshot().sha256 != before.sha256:
        raise RuntimeError("SOURCE_CHANGED_DURING_COPY")
    return snapshot_hash


class TrustedVisiblePytestAdapter:
    """Trust boundary: only curated real *visible* pytest check commands.

    A callable TestObservation executor for GP02C-02's controller. The controller
    alone charges the visible-test call and duration. This adapter must NOT also
    debit SharedBudget. Protected tests and reference fixes are inaccessible.
    """
    def __init__(self, *, source_root: str | Path, results_root: str | Path,
                 deadline: RealDeadline, pinned_environment_sha256: str,
                 python_executable: str | None = None,
                 trusted_environment: Mapping[str, str] | None = None,
                 stdout_max_bytes: int = 6000):
        self.source_root = Path(source_root).resolve(strict=True)
        self.results_root = Path(results_root).resolve()
        if self.source_root == self.results_root or self.source_root in self.results_root.parents or self.results_root in self.source_root.parents:
            raise ValueError("Evidence results must be outside the source workspace")
        self.results_root.mkdir(parents=True, exist_ok=True)
        self.deadline = deadline
        self.pinned_environment_sha256 = pinned_environment_sha256
        self.python_executable = str(Path(python_executable or sys.executable).resolve(strict=True))
        self.trusted_environment = frozen_pytest_environment(trusted_environment)
        self.stdout_max_bytes = stdout_max_bytes
        self.phase = "ordinary"
        self.counter = 0
        self.executions: list[dict] = []

    def _validate_check(self, check: VisibleCheck) -> None:
        if check.origin != "visible_repo":
            raise ValueError("Only visible-repository checks allowed")
        if check.pinned_environment_sha256 != self.pinned_environment_sha256:
            raise ValueError("Check environment not pinned to trusted adapter")
        argv = tuple(check.command)
        if (len(argv) < 5 or Path(argv[0]).resolve() != Path(self.python_executable)
            or argv[1:3] != ("-m", "pytest")):
            raise ValueError("Only Python -m pytest allowed, with frozen executable")
        if any(str(x).startswith("--junitxml") or x in {"--override-ini", "-c", "--pyargs"} for x in argv[3:]):
            raise ValueError("Trusted adapter owns the JUnit and pytest configuration")
        if any(x == ".." for token in argv[3:] for x in Path(token).parts):
            raise ValueError("No parent traversal in selected check")
        if not check.test_selection:
            raise ValueError("Explicit visible test selection required")
        if not any(check.test_selection == token or token.startswith(check.test_selection + "::") for token in argv[3:]):
            raise ValueError("Frozen test selection must occur in pytest arguments")

    def __call__(self, check: VisibleCheck) -> TestObservation:
        started = time.monotonic()
        self.counter += 1
        label = f"check_{self.counter:04d}"
        out_dir = self.results_root / label
        out_dir.mkdir(parents=True, exist_ok=False)
        stdout_path, stderr_path, junit_path = (out_dir / "stdout.txt", out_dir / "stderr.txt", out_dir / "junit.xml")
        before = WorkspaceFingerprinter(self.source_root).snapshot().sha256
        after = before
        snapshot_sha = None
        after_snapshot_sha = None
        raw_exit_code = None
        cancelled = False
        setup_error = False
        counts = (None,) * 5
        message = ""
        snapshot_dir = ""
        try:
            self._validate_check(check)
            if self.deadline.remaining(self.phase) <= 0:
                cancelled = True
                message = "ABSOLUTE_DEADLINE_EXPIRED_BEFORE_START"
            else:
                # Isolated temp directory is OUTSIDE the source/repo and results.
                with tempfile.TemporaryDirectory(prefix="gp02c03_snapshot_") as temp:
                    root = Path(temp) / "source"
                    snapshot_sha = copy_isolated_snapshot(self.source_root, root)
                    snapshot_dir = "ephemeral isolated source copy"
                    cwd = (root / check.working_dir).resolve(strict=True)
                    if root != cwd and root not in cwd.parents:
                        raise ValueError("Working directory escaped isolated snapshot")
                    env = dict(self.trusted_environment)
                    env["PYTHONPATH"] = str(root) + (os.pathsep + str(root / "src") if (root / "src").is_dir() else "")
                    env["TMPDIR"] = str(Path(temp))
                    env["TMP"] = str(Path(temp))
                    env["TEMP"] = str(Path(temp))
                    argv = tuple(check.command) + (f"--junitxml={junit_path}", "-p", "no:cacheprovider")
                    effective_timeout = self.deadline.remaining(self.phase)
                    if effective_timeout <= 0:
                        cancelled = True
                        message = "ABSOLUTE_DEADLINE_EXPIRED_AFTER_SNAPSHOT"
                    else:
                        raw_exit_code, cancelled, _exec_elapsed = cancelable_argv(
                            argv, cwd, env, stdout_path, stderr_path, effective_timeout)
                    after_snapshot_sha = WorkspaceFingerprinter(root).snapshot().sha256
                    if after_snapshot_sha != snapshot_sha:
                        setup_error = True
                        message = "TEST_MUTATED_ISOLATED_SOURCE"
                    if junit_path.is_file() and not cancelled:
                        try:
                            counts = junit_counts(junit_path)
                        except (ET.ParseError, ValueError, OSError) as exc:
                            setup_error = True
                            message = f"INVALID_JUNIT_{type(exc).__name__}"
                    elif not cancelled:
                        setup_error = True
                        message = "MISSING_JUNIT"
        except Exception as exc:
            setup_error = True
            message = f"ADAPTER_ERROR_{type(exc).__name__}: {str(exc)[:200]}"
        after = WorkspaceFingerprinter(self.source_root).snapshot().sha256
        if after != before or (snapshot_sha is not None and snapshot_sha != before):
            setup_error = True
            message = "LIVE_SOURCE_CHANGED_DURING_TEST"
        wall = time.monotonic() - started
        if self.deadline.remaining(self.phase) <= 0 and (cancelled or wall > 0):
            # Strict finish time check. When clock hits cutoff, no PASS is qualified.
            cancelled = True
            message = message or "ABSOLUTE_DEADLINE_REACHED"
        stdout = _read_head(stdout_path, self.stdout_max_bytes) if stdout_path.exists() else ""
        stderr = _read_head(stderr_path, self.stdout_max_bytes) if stderr_path.exists() else ""
        if message:
            stderr += ("\n" if stderr else "") + message
        observation = TestObservation(
            exit_code=raw_exit_code,
            tests_collected=counts[0], tests_passed=counts[1],
            tests_failed=counts[2], tests_errored=counts[3], tests_skipped=counts[4],
            elapsed_seconds=wall, stdout=stdout, stderr=stderr,
            timed_out=cancelled, setup_error=setup_error)
        audit = {
            "schema": SCHEMA, "check_id": check.check_id, "phase": self.phase,
            "command_sha256": check.command_sha256, "selection": check.test_selection,
            "definition_sha256": check.definition_sha256,
            "environment_sha256": self.pinned_environment_sha256,
            "source_sha256_before": before, "source_sha256_after": after,
            "copied_snapshot_sha256": snapshot_sha, "snapshot_sha256_after_test": after_snapshot_sha,
            "snapshot_policy": "ephemeral copied source; chmod readonly; postcheck digest; not security hardened",
            "junit_sha256": sha256_file(junit_path) if junit_path.exists() else None,
            "stdout_file_sha256": sha256_file(stdout_path) if stdout_path.exists() else None,
            "stderr_file_sha256": sha256_file(stderr_path) if stderr_path.exists() else None,
            "raw_exit_code": raw_exit_code, "process_cancelled": cancelled,
            "observed_outcome": observation.observed_outcome,
            "elapsed_seconds": wall, "counts": list(counts),
            "error": message, "executed_utc": utc(),
        }
        (out_dir / "attestation.json").write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")
        self.executions.append({"audit_path": str(out_dir / "attestation.json"), **audit})
        return observation


class LiveVerificationBridge(VerificationController):
    """B/C controller with actual time caps and a single-charge test executor.

    Only trusted subprocesses are cancellable here; ADK model requests and other
    agent tools must be routed via the equivalent hard-deadline worker in GP02C-04.
    """
    def __init__(self, ledger: EvidenceLedger, budget: SharedBudget,
                 executor: TrustedVisiblePytestAdapter,
                 production_patch_sha256: Callable[[], str], **kwargs):
        super().__init__(ledger, budget, executor, production_patch_sha256, **kwargs)
        self.deadline = executor.deadline

    def _qualify_receipt(self, receipt, *, check_id: str,
                         provenance: str, allowed_seconds: float) -> None:
        super()._qualify_receipt(receipt, check_id=check_id,
                                 provenance=provenance, allowed_seconds=allowed_seconds)
        phase = "proactive" if provenance == "controller_recheck" else (
            "final" if provenance == "final_gate" else "ordinary")
        if receipt.timed_out or self.deadline.remaining(phase) <= 0:
            if receipt.receipt_id not in self.ledger.unqualified_receipts:
                self.ledger.disqualify_receipt(receipt, "ABSOLUTE_DEADLINE")
                self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                           "check_id": check_id, "provenance": provenance,
                                           "reason": "absolute_deadline"})

    def _run_check(self, check_id: str, provenance: str):
        phase = "proactive" if provenance == "controller_recheck" else (
            "final" if provenance == "final_gate" else "ordinary")
        if self.deadline.remaining(phase) <= 0:
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET", "check_id": check_id,
                                       "provenance": provenance, "reason": "absolute_deadline"})
            return None
        old = self.executor.phase
        self.executor.phase = phase
        try:
            receipt = super()._run_check(check_id, provenance)
        finally:
            self.executor.phase = old
        return receipt

    def observe_agent_action(self, kind: str, action: Callable[[], object], **kwargs) -> dict:
        if self.deadline.remaining("ordinary") <= 0:
            raise RuntimeError("Absolute repair deadline exhausted")
        start = time.monotonic()
        budget_before = self.budget.used_wall_seconds
        try:
            return super().observe_agent_action(kind, action, elapsed_seconds=0.0)
        finally:
            # super has already charged all automatic checks; charge only action
            # and wrapper overhead (not another tool call, not double testing).
            wall = time.monotonic() - start
            already_charged = self.budget.used_wall_seconds - budget_before
            self.budget.used_wall_seconds += max(0.0, wall - already_charged)
            if self.deadline.remaining("ordinary") <= 0:
                self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                           "reason": "repair_deadline_reached_during_action"})

    def final_gate(self) -> GateDecision:
        if self.deadline.remaining("final") <= 0:
            return GateDecision(False, "UNVERIFIED_BUDGET", {}, 0)
        decision = super().final_gate()
        if (not decision.accepted and any(e.get("type") == "UNVERIFIED_BUDGET"
                                         and e.get("reason") == "absolute_deadline"
                                         for e in self.ledger.events)):
            return GateDecision(False, "UNVERIFIED_BUDGET", decision.checks, decision.attempts)
        if decision.accepted and self.deadline.remaining("final") <= 0:
            return GateDecision(False, "UNVERIFIED_BUDGET", decision.checks, decision.attempts)
        return decision

    def verify_delivery_identity(self, approved: GateDecision) -> bool:
        """Recheck both full workspace and trusted production-patch identities."""
        if not approved.accepted:
            return False
        self.ledger.observe_workspace("delivery_identity_sync")
        return (self.ledger.snapshot.sha256 == approved.accepted_workspace_sha256
                and self.ledger.revision == approved.accepted_revision
                and self._patch_digest() == approved.accepted_patch_sha256
                and self.deadline.remaining("final") > 0)


def digest_production_patch(source_root: str | Path, baseline_root: str | Path,
                            *, include: tuple[str, ...] = ("app.py",)) -> str:
    """Small CPU-fixture patch identity; production ADK must supply git-style patch.

    Uses source+baseline content with explicit additions/deletions; excludes tests.
    """
    source, baseline = Path(source_root), Path(baseline_root)
    entries = []
    for name in include:
        before, after = baseline / name, source / name
        old_sha = sha256_file(before) if before.is_file() else None
        new_sha = sha256_file(after) if after.is_file() else None
        if old_sha != new_sha:
            entries.append((name, old_sha, new_sha))
    return canonical_hash(entries)


Writing gp02c_live_adapter.py


**Observation after cell 03:** `gp02c_live_adapter.py` should be created without exceptions. This cell only writes source code; no tests or model calls execute.

## 04 — Implement 18 live-adapter smoke tests (including actual pytest subprocesses)

Test actual pytest PASS/FAIL/SKIP/zero-collection, B/C rechecks, source isolation, absolute timeout, final reserve, manifest pinning, read-only policy, and exact delivery binding.

In [5]:
%%writefile test_gp02c_live_adapter.py
"""Isolated real-pytest CPU smoke tests for GP02C-03; never touch held-out tasks."""
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import json
import sys
import time
import unittest

from gp02c_ledger import (
    EvidenceLedger, SharedBudget, VisibleCheck, VisibleManifest,
    canonical_hash,
)
from gp02c_live_adapter import (
    LiveVerificationBridge, RealDeadline, TrustedVisiblePytestAdapter,
    digest_production_patch, frozen_pytest_environment, junit_counts,
    observed_environment_identity,
)


class LiveAdapterSmoke(unittest.TestCase):
    def setUp(self):
        self.tmp = TemporaryDirectory(prefix="gp02c03_unit_")
        self.addCleanup(self.tmp.cleanup)
        base = Path(self.tmp.name)
        self.root = base / "source"
        self.baseline = base / "baseline"
        self.results = base / "evidence"
        for d in (self.root, self.baseline):
            (d / "tests").mkdir(parents=True)
        self._write_source("VALUE = 1\n")
        self._write_test("from app import VALUE\ndef test_value():\n    assert VALUE == 1\n")
        self.identity, self.profile = observed_environment_identity()
        self.check = VisibleCheck(
            check_id="visible-value", command=(sys.executable, "-m", "pytest", "-q", "tests/test_visible.py"),
            definition_sha256=canonical_hash("frozen:visible-value:v1"),
            reserve_seconds=0.01, test_selection="tests/test_visible.py", expected_collected=1,
            pinned_environment_sha256=self.identity,
        )
        self.manifest = VisibleManifest((self.check,), strict=True)

    def _write_source(self, value):
        for d in (self.root, self.baseline):
            (d / "app.py").write_text(value)

    def _write_test(self, value):
        for d in (self.root, self.baseline):
            (d / "tests" / "test_visible.py").write_text(value)

    def setup_run(self, arm="B", repair_cutoff=16.0, overall_cutoff=25.0,
                  expected_collected=1, max_tools=12, profile=None):
        env_id = self.identity
        check = replace(self.check, expected_collected=expected_collected)
        manifest = VisibleManifest((check,), strict=True)
        deadline = RealDeadline(repair_cutoff, overall_cutoff)
        ledger = EvidenceLedger(self.root, manifest, run_id=f"demo-{arm}", task_id="synthetic_01",
                                arm=arm, seed=20261105, environment_sha256=env_id)
        budget = SharedBudget(max_wall_seconds=overall_cutoff, max_tool_calls=max_tools,
                              final_reserve_seconds=overall_cutoff-repair_cutoff)
        adapter = TrustedVisiblePytestAdapter(
            source_root=self.root, results_root=self.results, deadline=deadline,
            pinned_environment_sha256=env_id)
        controller = LiveVerificationBridge(ledger, budget, adapter,
                                            lambda: digest_production_patch(self.root, self.baseline))
        return ledger, budget, adapter, controller

    def test_01_real_pytest_pass_creates_junit_and_valid_receipt(self):
        ledger, budget, adapter, ctl = self.setup_run()
        decision = ctl.final_gate()
        self.assertTrue(decision.accepted, decision)
        self.assertEqual(ledger.current_status("visible-value"), "CURRENT_PASS")
        self.assertEqual(len(ledger.receipts), 1)
        self.assertEqual(budget.used_tool_calls, 1)
        self.assertEqual(budget.used_test_runs, 1)
        audit = json.loads(Path(adapter.executions[0]["audit_path"]).read_text())
        self.assertEqual(audit["counts"], [1, 1, 0, 0, 0])
        self.assertEqual(audit["source_sha256_before"], audit["copied_snapshot_sha256"])
        self.assertEqual(audit["source_sha256_before"], audit["source_sha256_after"])
        self.assertEqual(audit["copied_snapshot_sha256"], audit["snapshot_sha256_after_test"])
        self.assertIsNotNone(audit["junit_sha256"])
        self.assertTrue(ctl.verify_delivery_identity(decision))

    def test_02_B_final_rechecks_after_source_edit(self):
        ledger, budget, adapter, ctl = self.setup_run("B")
        ctl._run_check("visible-value", "agent")
        (self.root / "app.py").write_text("VALUE = 2\n")
        ledger.observe_workspace("edit")
        self.assertEqual(ledger.current_status("visible-value"), "STALE")
        decision = ctl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual(decision.status, "VISIBLE_CHECK_FAILED")
        self.assertEqual(ledger.current_status("visible-value"), "CURRENT_FAIL")
        self.assertEqual(budget.used_tool_calls, 2)

    def test_03_C_post_edit_rechecks_and_records_real_regression(self):
        ledger, budget, adapter, ctl = self.setup_run("C")
        ctl._run_check("visible-value", "agent")
        result = ctl.observe_agent_action("edit_file", lambda: (self.root / "app.py").write_text("VALUE = 2\n"))
        feedback = ctl.agent_visible_feedback(result)
        self.assertEqual(feedback["rechecks"][0]["status"], "CURRENT_FAIL")
        self.assertEqual(budget.used_test_runs, 2)
        self.assertEqual(budget.used_tool_calls, 3)  # initial test, edit, post-edit test
        self.assertTrue(any(e["type"] == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT" for e in ledger.events))
        self.assertEqual(len(adapter.executions), 2)
        self.assertNotIn("stdout", json.dumps(feedback))

    def test_04_C_post_edit_rechecks_harmless_change(self):
        ledger, budget, adapter, ctl = self.setup_run("C")
        ctl._run_check("visible-value", "agent")
        result = ctl.observe_agent_action("edit_file", lambda: (self.root / "app.py").write_text("VALUE = 1\n# comment\n"))
        self.assertEqual(ctl.agent_visible_feedback(result)["rechecks"][0]["status"], "CURRENT_PASS")
        decision = ctl.final_gate()
        self.assertTrue(decision.accepted)
        self.assertEqual(budget.used_test_runs, 2)  # no redundant final rerun
        self.assertEqual(budget.used_tool_calls, 3)

    def test_05_all_skipped_fails_positive_evidence_requirement(self):
        self._write_test("import pytest\n@pytest.mark.skip\ndef test_ignored():\n    assert True\n")
        ledger, budget, adapter, ctl = self.setup_run()
        decision = ctl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual(ledger.receipts[-1].observed_outcome, "ERROR")
        self.assertEqual(ledger.receipts[-1].tests_skipped, 1)

    def test_06_collection_mismatch_does_not_verify(self):
        ledger, budget, adapter, ctl = self.setup_run(expected_collected=2)
        decision = ctl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual(ledger.receipts[-1].qualification_error, "UNEXPECTED_COLLECTION")

    def test_07_zero_collected_does_not_verify(self):
        self._write_test("VALUE = 99\n")
        ledger, budget, adapter, ctl = self.setup_run()
        decision = ctl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertNotEqual(ledger.current_status("visible-value"), "CURRENT_PASS")

    def test_08_actual_timeout_cancels_real_pytest(self):
        self._write_test("import time\ndef test_slow():\n    time.sleep(3)\n    assert True\n")
        ledger, budget, adapter, ctl = self.setup_run(repair_cutoff=0.20, overall_cutoff=0.85)
        t0 = time.monotonic()
        decision = ctl.final_gate()
        total = time.monotonic() - t0
        self.assertFalse(decision.accepted)
        self.assertEqual(decision.status, "UNVERIFIED_BUDGET")
        self.assertLess(total, 2.0, f"Real timeout not enforced: {total:.2f}s")
        self.assertTrue(adapter.executions[-1]["process_cancelled"])
        self.assertEqual(ledger.receipts[-1].observed_outcome, "ERROR")
        self.assertIn(ledger.receipts[-1].receipt_id, ledger.unqualified_receipts)

    def test_09_repair_reserve_blocks_proactive_pytest(self):
        ledger, budget, adapter, ctl = self.setup_run("C", repair_cutoff=0.10, overall_cutoff=5.0)
        time.sleep(0.14)
        r = ctl._run_check("visible-value", "controller_recheck")
        self.assertIsNone(r)
        self.assertEqual(len(adapter.executions), 0)
        self.assertEqual(budget.used_test_runs, 0)
        self.assertGreater(ctl.deadline.remaining("final"), 0)

    def test_10_patch_identity_and_full_workspace_identity_both_required(self):
        ledger, budget, adapter, ctl = self.setup_run()
        decision = ctl.final_gate()
        self.assertTrue(decision.accepted)
        (self.root / "runtime.cfg").write_text("changed=true\n")
        self.assertFalse(ctl.verify_delivery_identity(decision))
        self.assertEqual(digest_production_patch(self.root, self.baseline), decision.accepted_patch_sha256)

    def test_11_test_running_on_isolated_copy_does_not_modify_live_workspace(self):
        self._write_test("from pathlib import Path\ndef test_mutation():\n    Path('app.py').write_text('VALUE = 999\\n')\n    assert True\n")
        ledger, budget, adapter, ctl = self.setup_run()
        decision = ctl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual((self.root / "app.py").read_text(), "VALUE = 1\n")
        self.assertNotEqual(ledger.current_status("visible-value"), "CURRENT_PASS")

    def test_12_invalid_untrusted_command_rejected(self):
        ledger, budget, adapter, ctl = self.setup_run()
        invalid = replace(self.check, command=("bash", "-c", "echo hello"))
        ob = adapter(invalid)
        self.assertEqual(ob.observed_outcome, "ERROR")
        self.assertTrue(ob.setup_error)

    def test_13_bad_environment_pin_rejected(self):
        ledger, budget, adapter, ctl = self.setup_run()
        invalid = replace(self.check, pinned_environment_sha256="0" * 64)
        ob = adapter(invalid)
        self.assertEqual(ob.observed_outcome, "ERROR")
        self.assertTrue(ob.setup_error)

    def test_14_junit_malformed_and_skip_count_parsing(self):
        fake = Path(self.tmp.name) / "junit.xml"
        fake.write_text('<testsuite><testcase name="ok"/><testcase name="skip"><skipped/></testcase></testsuite>')
        self.assertEqual(junit_counts(fake), (2, 1, 0, 0, 1))
        fake.write_text("<testsuite><broken")
        with self.assertRaises(Exception):
            junit_counts(fake)

    def test_15_real_deadline_continuous_not_per_test_reset(self):
        now = [100.0]
        d = RealDeadline(840, 900, started_monotonic=100.0, now=lambda: now[0])
        self.assertEqual(d.remaining("proactive"), 840)
        now[0] = 940.0
        self.assertEqual(d.remaining("proactive"), 0)
        self.assertEqual(d.remaining("final"), 60)
        now[0] = 1001.0
        self.assertEqual(d.remaining("final"), 0)

    def test_16_strict_source_symlink_rejected_no_external_follow(self):
        ext = Path(self.tmp.name) / "outside.txt"
        ext.write_text("secret")
        (self.root / "link.py").symlink_to(ext)
        ledger, budget, adapter, ctl = self.setup_run()
        decision = ctl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual(ledger.receipts[-1].observed_outcome, "ERROR")
        self.assertTrue((self.root / "link.py").is_symlink())

    def test_17_working_directory_traversal_is_rejected(self):
        with self.assertRaises(ValueError):
            replace(self.check, working_dir="../secret")

    def test_18_live_workspace_change_detected_as_stale(self):
        ledger, budget, adapter, ctl = self.setup_run()
        # Forcing a mutation during a real source-copy/test via a thread is
        # nondeterministic; verify the core out-of-band mutation detection.
        ctl._run_check("visible-value", "agent")
        (self.root / "app.py").write_text("VALUE = 3\n")
        self.assertEqual(ledger.current_status("visible-value"), "STALE")
        self.assertEqual(budget.used_test_runs, 1)


if __name__ == "__main__":
    unittest.main(verbosity=2)


Writing test_gp02c_live_adapter.py


**Observation after cell 04:** `test_gp02c_live_adapter.py` should be created without exceptions. This cell only writes source code; no tests or model calls execute.

## 05 — Execute contract tests and real-`pytest` smoke tests

This runs all **59 controller deterministic** tests plus **18 additional live-adapter tests (many of which launch actual subprocesses)**. The timeout test deliberately starts a slow `pytest` and cancels the process group. CPU execution can take tens of seconds depending on the machine.

In [6]:
import sys, subprocess, re
from pathlib import Path
cmd = [sys.executable, '-m', 'unittest', '-v', 'test_gp02c_ledger', 'test_gp02c_live_adapter']
result = subprocess.run(cmd, text=True, capture_output=True, check=False)
log = (result.stdout or '') + '\n' + (result.stderr or '')
Path('GP02C_03_CPU_Test_Output_v1_1.txt').write_text(log)
print('\n'.join(log.splitlines()[-35:]))
found = re.search(r'Ran (\d+) tests?', log)
if result.returncode != 0 or not found or not re.search(r'(?m)^OK$', log):
    raise RuntimeError('GP02C-03 CPU tests failed; review the saved output')
TESTS_PASSED = int(found.group(1))
if TESTS_PASSED != 77:
    raise RuntimeError(f'Expected 74 tests; observed {TESTS_PASSED}')
print(f'VALIDATED: {TESTS_PASSED}/77 = 59 controller + 18 live-adapter smoke tests (including actual pytest subprocesses)')

test_48_gate_final_allowed_recheck_success_is_accepted (test_gp02c_ledger.Fixtures.test_48_gate_final_allowed_recheck_success_is_accepted) ... ok
test_49_C_overrunning_proactive_test_cannot_qualify (test_gp02c_ledger.Fixtures.test_49_C_overrunning_proactive_test_cannot_qualify) ... ok
test_50_no_patch_infra_failure_retains_classification (test_gp02c_ledger.Fixtures.test_50_no_patch_infra_failure_retains_classification) ... ok
test_51_same_patch_digest_different_workspace_fails_delivery (test_gp02c_ledger.Fixtures.test_51_same_patch_digest_different_workspace_fails_delivery) ... ok
test_52_missing_delivery_workspace_is_never_accepted (test_gp02c_ledger.Fixtures.test_52_missing_delivery_workspace_is_never_accepted) ... ok
test_53_retry_requires_same_config_env_and_evidence (test_gp02c_ledger.Fixtures.test_53_retry_requires_same_config_env_and_evidence) ... ok
test_54_last_edit_after_gate_stales_receipt_and_export_refreshes (test_gp02c_ledger.Fixtures.test_54_last_edit_after_gate_stales_r

**Cell 05 observation:** `Ran 77 tests ... OK` establishes controller regression compliance plus real Python subprocess test execution. It **does not** establish that any Gemma coding task is solved or that the held-out cohort has been evaluated. Tests marked as intentional negative cases count as passing *the smoke-test expectation*.

## 06 — Real A/B/C policy illustration (one selected visible pytest check)

Each arm uses an independent synthetic workspace. A selected test passes; we then make an innocuous source edit. Arm A leaves the prior pass stale; B runs the same final gate as C at submission; C rechecks after the edit and can reuse its current pass at the final gate.

In [7]:
from pathlib import Path
from tempfile import TemporaryDirectory
from gp02c_ledger import EvidenceLedger, VisibleCheck, VisibleManifest, SharedBudget, canonical_hash
from gp02c_live_adapter import TrustedVisiblePytestAdapter, LiveVerificationBridge, RealDeadline, observed_environment_identity, digest_production_patch

DEMO_ROWS = []
for arm in ('A','B','C'):
    with TemporaryDirectory(prefix='gp02c03_demo_') as d:
        base = Path(d)
        workspace, baseline, evidence = base / 'workspace', base / 'baseline', base / 'evidence'
        for r in (workspace,baseline):
            (r/'tests').mkdir(parents=True)
            (r/'app.py').write_text('VALUE = 1\n')
            (r/'tests'/'test_visible.py').write_text('from app import VALUE\ndef test_value():\n    assert VALUE == 1\n')
        env_sha, env_details = observed_environment_identity()
        check=VisibleCheck('visible-smoke', (sys.executable,'-m','pytest','-q','tests/test_visible.py'),
           canonical_hash('test_visible.py:1'), reserve_seconds=0.01,
           test_selection='tests/test_visible.py', expected_collected=1,
           pinned_environment_sha256=env_sha)
        manifest=VisibleManifest((check,), strict=True)
        ledger=EvidenceLedger(workspace,manifest,run_id=f'smoke-{arm}',task_id='synthetic_01',arm=arm,
                              seed=20261105,environment_sha256=env_sha)
        budget=SharedBudget(max_wall_seconds=45,max_tool_calls=15,final_reserve_seconds=10)
        adapter=TrustedVisiblePytestAdapter(source_root=workspace,results_root=evidence,
                  deadline=RealDeadline(35,45),pinned_environment_sha256=env_sha)
        bridge=LiveVerificationBridge(ledger,budget,adapter,
                  lambda:digest_production_patch(workspace,baseline))
        first=bridge._run_check('visible-smoke','agent')
        edit=bridge.observe_agent_action('edit_file',lambda:(workspace/'app.py').write_text('VALUE = 1\n# harmless edit\n'))
        after_edit=ledger.current_status('visible-smoke')
        verdict='NOT_APPLICABLE'
        if arm != 'A':
            gate=bridge.final_gate()
            verdict=gate.status
            if not gate.accepted or not bridge.verify_delivery_identity(gate):
                raise RuntimeError(f'{arm} live-policy demonstration did not verify')
        DEMO_ROWS.append({'arm':arm,'first_test':'PASS' if first.observed_outcome=='PASS' else 'FAIL',
           'after_edit':after_edit, 'final_gate':verdict,
           'real_pytest_executions':budget.used_test_runs, 'total_tool_calls':budget.used_tool_calls})
print('Arm | Before edit | After edit | Final gate | pytest runs | tool calls')
for row in DEMO_ROWS:
    print(f"{row['arm']}   | {row['first_test']:11} | {row['after_edit']:12} | {row['final_gate']:16} | {row['real_pytest_executions']} | {row['total_tool_calls']}")
if [x['real_pytest_executions'] for x in DEMO_ROWS] != [1,2,2]:
    raise RuntimeError('A/B/C subprocess counts disagree with the frozen contract')

Arm | Before edit | After edit | Final gate | pytest runs | tool calls
A   | PASS        | STALE        | NOT_APPLICABLE   | 1 | 2
B   | PASS        | STALE        | VERIFIED_VISIBLE | 2 | 3
C   | PASS        | CURRENT_PASS | VERIFIED_VISIBLE | 2 | 3


**Cell 06 interpretation:** The A/B/C timing differences are observed here using **real tests**, unlike GP02C-02's synthetic receipts. This is still a constructed demonstration using harmless edits, **not** repair-success evidence. Note that A has no mandatory final gate, B rechecks at submission, and C rechecks after the edit.

## 07 — Export CPU-only evidence

Exports the updated controller and live adapter, **59+18 CPU tests**, logs, and report in a ZIP. No protected evaluation task data is included.


In [8]:
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib, json, sys
files = [Path(x) for x in ('gp02c_ledger.py','test_gp02c_ledger.py',
         'gp02c_live_adapter.py','test_gp02c_live_adapter.py','GP02C_03_CPU_Test_Output_v1_1.txt')]
report = {
 'stage':'GP02C-03', 'version':'1.1', 'scope':'isolated synthetic real-pytest CPU smoke',
 'executed_utc':datetime.now(timezone.utc).isoformat(),
 'test_cases':TESTS_PASSED, 'deterministic_controller_tests':59,
 'live_adapter_smoke_tests':18, 'unit_test_return_code':result.returncode,
 'gpu_calls':0, 'gemma_calls':0, 'adk_live_wired':False, 'protected_evaluator_calls':0,
 'real_policy_demo':DEMO_ROWS,
 'remaining': ['ADK tool/mode wiring', 'model call hard deadline', 'production wheel hashes',
               'repository-specific visible-check manifests', 'development paired pilot'],
 'files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
}
report_path=Path('GP02C_03_CPU_Validation_Results_v1_1.json')
report_path.write_text(json.dumps(report,indent=2)+'\n')
zip_path=Path('GP02C_03_CPU_Evidence_v1_1.zip')
with ZipFile(zip_path,'w',ZIP_DEFLATED) as z:
    for p in files+[report_path]:z.write(p,p.name)
print('Saved:',report_path,zip_path)
print('TESTS:', report['test_cases'], '| Real-process smoke cases:', report['live_adapter_smoke_tests'])

Saved: GP02C_03_CPU_Validation_Results_v1_1.json GP02C_03_CPU_Evidence_v1_1.zip
TESTS: 77 | Real-process smoke cases: 18


## Completion boundary and next step

**GP02C-03 passes** only if all original and new tests pass and the notebook exports its evidence ZIP. Real subprocess deadlines and source snapshots are now demonstrated on CPU. **Do not claim full ADK integration:** GP02C-04 must hook this adapter into the exact v2.7 worker APIs, implement worker-model cancellation, pin a repository-compatible visible-check manifest and run paired B/C development pilots. Keep the 36-task held-out evaluation untouched until those gates are met. The six GP02B development traces currently have **zero confirmed explicit-pytest PASS followed by a successful production edit**, making the exposure pilot a prerequisite, not an optional refinement.